# Advanced Statistical and Machine-Learning Analysis of Sheep Methane Phenotypes
**Publication-oriented analysis notebook | LMD monitoring rounds 2024–2025**

This notebook analyses animal-level methane monitoring records to address:

1. Identification of the most defensible methane phenotype(s) for representing enteric methane emissions.
2. Predictability of methane phenotypes from animal, management and monitoring covariates.
3. Differences between CBBP and Non-CBBP sheep.
4. Classification of high versus low methane emitters.
5. Natural methane emission patterns across animals and monitoring conditions.
6. Comparison of conventional statistical methods with machine-learning and exploratory time-series methods (ARIMA, Prophet and LSTM).

**Data unit:** one animal monitoring session/record. The supplied file contains summary features, not the original second-by-second methane trace. Accordingly, time-series methods below operate only on repeated session-level observations or aggregate trends; they must not be interpreted as within-session sensor-signal models.

**Reproducibility:** set a fixed random seed; retain animal identifiers for grouped validation; export tables, figures and model artifacts to `outputs/`. All inferential findings must be interpreted alongside effect sizes, confidence intervals, diagnostics and the sampling design.

## 0. Environment and dependencies

Install packages if needed. Core analyses use pandas, numpy, scipy, statsmodels, scikit-learn, matplotlib and seaborn. Prophet and TensorFlow are optional and are handled gracefully if unavailable.

In [ ]:
# Uncomment in a fresh environment:
# %pip install -q pandas numpy scipy statsmodels scikit-learn matplotlib seaborn openpyxl joblib
# %pip install -q prophet tensorflow  # Optional; may require platform-specific setup

import os, re, json, warnings, math, itertools, textwrap, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from statsmodels.formula.api import ols, mixedlm, logit
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.graphics.gofplots import qqplot
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (r2_score, mean_absolute_error, mean_squared_error,
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, RocCurveDisplay, PrecisionRecallDisplay)
from sklearn.ensemble import (RandomForestRegressor, ExtraTreesRegressor,
    RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingRegressor,
    HistGradientBoostingClassifier, RandomForestRegressor)
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.svm import SVR, SVC
from sklearn.dummy import DummyRegressor, DummyClassifier
import statsmodels.api as sm
import joblib

SEED = 42
np.random.seed(SEED)
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
OUT = Path("outputs")
FIG = OUT / "figures"
TAB = OUT / "tables"
MOD = OUT / "models"
for p in [OUT, FIG, TAB, MOD]:
    p.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version)
print("Output directory:", OUT.resolve())

## 1. Load data and validate the schema

Set `DATA_PATH` to the uploaded CSV or place it beside this notebook. The notebook does not assume that all rows are independent.

In [ ]:
DATA_PATH = Path("20261112_OldLMD_Phenotype_Data_RO.csv")
if not DATA_PATH.exists():
    # Common notebook environment fallback; change this path if necessary.
    DATA_PATH = Path("/mnt/data/20261112_OldLMD_Phenotype_Data_RO.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError("Set DATA_PATH to the supplied CSV file.")

df_raw = pd.read_csv(DATA_PATH)
df = df_raw.copy()
print("Shape:", df.shape)
display(df.head())

In [ ]:
expected = [
"AnimalID","Sex","Breed","weight","Round","log_day","Description","Member",
"Diurnality","feedtime","Assessor","file_date","recording_duration","n_peaks",
"n_resp","n_eruct","mean_methane","mean_resp","mean_eruct","max_resp",
"max_eruct","peak_duration_sec","eruct_per_min","AUC_peaks","AUC_peaks_3min",
"AUC_peak_per_sec","n_records","full_duration_sec","AUC_full","AUC_3min","AUC_per_sec"
]
missing_cols = sorted(set(expected) - set(df.columns))
extra_cols = sorted(set(df.columns) - set(expected))
print("Missing expected columns:", missing_cols)
print("Additional columns:", extra_cols)
assert not missing_cols, "Required columns missing; inspect source file."

## 2. Data cleaning, harmonisation and quality control

The code standardises strings and dates, checks duplicate animal-session records, identifies impossible/non-positive values and creates derived duration-normalised variables. No biological observations are deleted automatically. Review the QC outputs before deciding whether to exclude any record.

In [ ]:
# Standardise whitespace and common Unicode dash variants in text fields.
for c in df.select_dtypes(include="object").columns:
    df[c] = df[c].astype("string").str.strip()
    df[c] = df[c].str.replace(r"\s+", " ", regex=True)

df["file_date"] = pd.to_datetime(df["file_date"], errors="coerce")
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
for c in numeric_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Harmonise round labels without changing their meaning.
df["Round"] = df["Round"].replace({"Round 1":"R1","Round 2":"R2","1":"R1","2":"R2"})
df["Member"] = df["Member"].str.upper().replace({"NON CBBP":"NON-CBBP","NONCBBP":"NON-CBBP"})
df["Sex"] = df["Sex"].str.upper()
df["Description"] = df["Description"].str.title()
df["Breed"] = df["Breed"].str.title()

# One row is expected per animal per measurement day/session.
session_key = ["AnimalID","Round","file_date","log_day"]
duplicate_rows = df[df.duplicated(session_key, keep=False)].sort_values(session_key)
print("Duplicate rows for session key:", len(duplicate_rows))
display(duplicate_rows.head(20))

# QC flags, not automatic exclusions.
qc = pd.DataFrame(index=df.index)
qc["missing_animal_id"] = df["AnimalID"].isna()
qc["missing_date"] = df["file_date"].isna()
qc["nonpositive_weight"] = df["weight"].le(0) | df["weight"].isna()
qc["nonpositive_duration"] = df["full_duration_sec"].le(0) | df["full_duration_sec"].isna()
qc["negative_counts"] = df[["n_peaks","n_resp","n_eruct","n_records"]].lt(0).any(axis=1)
qc["negative_auc"] = df[["AUC_peaks","AUC_peaks_3min","AUC_full","AUC_3min"]].lt(0).any(axis=1)
qc["log_day_outside_1_3"] = ~df["log_day"].isin([1,2,3])
qc["duration_over_3min"] = df["full_duration_sec"] > 180
display(qc.sum().rename("flagged_rows").to_frame())
qc.to_csv(TAB/"quality_control_flags.csv", index=False)

In [ ]:
# Missingness and descriptive coverage.
missingness = (df.isna().mean()*100).sort_values(ascending=False).rename("missing_percent").to_frame()
display(missingness.round(2))
missingness.to_csv(TAB/"missingness.csv")

overview = {
    "n_records": len(df),
    "n_animals": df["AnimalID"].nunique(),
    "date_min": str(df["file_date"].min().date()),
    "date_max": str(df["file_date"].max().date()),
    "rounds": df["Round"].value_counts(dropna=False).to_dict(),
    "member": df["Member"].value_counts(dropna=False).to_dict(),
    "breed": df["Breed"].value_counts(dropna=False).to_dict(),
    "sex": df["Sex"].value_counts(dropna=False).to_dict()
}
print(json.dumps(overview, indent=2, default=str))
with open(OUT/"dataset_overview.json","w") as f: json.dump(overview,f,indent=2,default=str)

## 3. Phenotype construction and measurement validity

Candidate phenotypes include mean methane concentration (`mean_methane`), event-specific means, peak burden, and integrated methane exposure (`AUC_full`, `AUC_3min`, `AUC_peaks`). The AUC measures are duration-dependent unless normalised. The per-second variables may be more comparable across unequal monitoring durations, but their exact units and computation should be confirmed with the instrument protocol.

**Important:** do not call ppm·m values methane mass or methane yield (e.g., g/day). They are instrument-derived concentration/signal metrics unless calibrated against a validated flux standard.

In [ ]:
# Candidate outcomes and derived rates. Guard against zero denominators.
df["methane_rate_from_auc"] = df["AUC_full"] / df["full_duration_sec"].replace(0,np.nan)
df["peak_rate_per_min"] = df["n_peaks"] / (df["full_duration_sec"]/60).replace(0,np.nan)
df["resp_rate_per_min"] = df["n_resp"] / (df["full_duration_sec"]/60).replace(0,np.nan)
df["eruct_rate_per_min_recalc"] = df["n_eruct"] / (df["full_duration_sec"]/60).replace(0,np.nan)
df["auc_full_per_min"] = df["AUC_full"] / (df["full_duration_sec"]/60).replace(0,np.nan)
df["auc_peaks_per_min"] = df["AUC_peaks"] / (df["full_duration_sec"]/60).replace(0,np.nan)

phenotypes = [
"mean_methane","mean_resp","mean_eruct","max_resp","max_eruct",
"eruct_per_min","AUC_peaks","AUC_peaks_3min","AUC_peak_per_sec",
"AUC_full","AUC_3min","AUC_per_sec","methane_rate_from_auc",
"peak_rate_per_min","resp_rate_per_min","eruct_rate_per_min_recalc"
]
phenotypes = [c for c in phenotypes if c in df.columns]
desc = df[phenotypes].describe(percentiles=[.05,.25,.5,.75,.95]).T
display(desc.round(3))
desc.to_csv(TAB/"phenotype_descriptive_statistics.csv")

In [ ]:
# Distribution and outlier visualisation; extreme values are flagged, not deleted.
for col in ["mean_methane","AUC_full","AUC_per_sec","AUC_peaks","eruct_per_min"]:
    if col in df:
        fig, ax = plt.subplots(figsize=(7,4))
        sns.histplot(df[col].dropna(), kde=True, ax=ax)
        ax.set_title(f"Distribution of {col}")
        fig.tight_layout(); fig.savefig(FIG/f"distribution_{col}.png", dpi=300); plt.show()

outlier_rows=[]
for col in phenotypes:
    x=df[col].dropna()
    if len(x)>0:
        q1,q3=x.quantile([.25,.75]); iqr=q3-q1
        lo,hi=q1-1.5*iqr,q3+1.5*iqr
        outlier_rows.append({"variable":col,"n_outliers_iqr":int(((x<lo)|(x>hi)).sum()),
                             "lower_fence":lo,"upper_fence":hi})
outlier_table=pd.DataFrame(outlier_rows)
display(outlier_table)
outlier_table.to_csv(TAB/"potential_outliers_iqr.csv",index=False)

## 4. Which methane phenotype best represents emissions?

There is no external gold-standard emission measurement in this file. Therefore, this analysis ranks *candidate proxy phenotypes* using (i) repeatability across repeated observations, (ii) association with other methane indicators, (iii) robustness to monitoring duration, and (iv) ability to distinguish animal-level variation. This is a construct-validity assessment, not validation against methane mass flux.

### 4.1 Correlation and redundancy

In [ ]:
corr_vars=[c for c in phenotypes if df[c].notna().sum()>10]
corr=df[corr_vars].corr(method="spearman")
fig,ax=plt.subplots(figsize=(13,10))
sns.heatmap(corr,cmap="vlag",center=0,vmin=-1,vmax=1,annot=False,ax=ax)
ax.set_title("Spearman correlations among methane phenotype candidates")
fig.tight_layout(); fig.savefig(FIG/"phenotype_spearman_correlation.png",dpi=300); plt.show()
corr.to_csv(TAB/"phenotype_spearman_correlations.csv")

### 4.2 Within-animal repeatability

For animals with repeated observations, estimate a simple intraclass correlation coefficient (ICC) using a random-intercept model. This is a session-level repeatability estimate and can be affected by round, day, feeding and other systematic factors. The adjusted mixed model below includes available covariates where estimable.

In [ ]:
def estimate_icc(data, outcome, group="AnimalID"):
    z=data[[group,outcome]].dropna()
    if z[group].nunique()<2 or len(z)<10: return np.nan, None
    try:
        m=mixedlm(f"{outcome} ~ 1", z, groups=z[group]).fit(reml=True, method="lbfgs", disp=False)
        var_animal=float(m.cov_re.iloc[0,0])
        var_resid=float(m.scale)
        icc=var_animal/(var_animal+var_resid) if var_animal+var_resid>0 else np.nan
        return icc,m
    except Exception as e:
        return np.nan,str(e)

icc_results=[]
for col in phenotypes:
    icc,fit=estimate_icc(df,col)
    icc_results.append({"phenotype":col,"ICC_animal":icc,
                        "n_animals_repeated":int(df.groupby("AnimalID").filter(lambda x:len(x)>1)["AnimalID"].nunique())})
icc_table=pd.DataFrame(icc_results).sort_values("ICC_animal",ascending=False)
display(icc_table)
icc_table.to_csv(TAB/"phenotype_repeatability_ICC.csv",index=False)

### 4.3 Duration sensitivity

Compare raw AUC with duration-normalised AUC. If rankings or associations change materially, duration is an important measurement-design factor. The final phenotype recommendation should consider interpretability and repeatability, not select the variable with the largest numeric scale.

In [ ]:
duration_sensitivity=[]
for a,b in [("AUC_full","AUC_per_sec"),("AUC_peaks","AUC_peak_per_sec"),
            ("AUC_3min","AUC_3min")]:
    if a in df and b in df:
        z=df[[a,b]].dropna()
        rho,p=stats.spearmanr(z[a],z[b]) if len(z)>2 else (np.nan,np.nan)
        duration_sensitivity.append({"raw":a,"normalised":b,"n":len(z),"spearman_rho":rho,"p_value":p})
display(pd.DataFrame(duration_sensitivity))
pd.DataFrame(duration_sensitivity).to_csv(TAB/"duration_sensitivity.csv",index=False)

## 5. CBBP versus Non-CBBP differences

Primary outcomes are pre-specified here as `mean_methane`, `AUC_per_sec`, `AUC_full`, `eruct_per_min`, and `mean_eruct`. Report unadjusted group summaries and adjusted mixed-effects models with animal random intercepts. Include breed, sex, body weight, age/group, round, diurnality and feed-time as covariates when supported by the data. `Description` contains several categories (e.g., Adult, Ewe, Lamb); it is not necessarily a clean age variable, so interpret accordingly.

For skewed outcomes, log transformation may improve model residuals. Effect sizes and confidence intervals matter more than p-values alone.

In [ ]:
primary_outcomes=[c for c in ["mean_methane","AUC_per_sec","AUC_full","eruct_per_min","mean_eruct"] if c in df]
group_summary=df.groupby("Member")[primary_outcomes].agg(["count","mean","std","median"])
display(group_summary)
group_summary.to_csv(TAB/"CBBP_nonCBBP_descriptive.csv")

# Welch tests and rank-based sensitivity analysis; BH correction across outcomes.
tests=[]
for y in primary_outcomes:
    a=df.loc[df.Member.eq("CBBP"),y].dropna()
    b=df.loc[df.Member.eq("NON-CBBP"),y].dropna()
    if min(len(a),len(b))>=3:
        t,p=stats.ttest_ind(a,b,equal_var=False)
        u,pu=stats.mannwhitneyu(a,b,alternative="two-sided")
        pooled=np.sqrt(((len(a)-1)*a.var(ddof=1)+(len(b)-1)*b.var(ddof=1))/(len(a)+len(b)-2))
        d=(a.mean()-b.mean())/pooled if pooled>0 else np.nan
        tests.append({"outcome":y,"n_CBBP":len(a),"n_nonCBBP":len(b),
                      "mean_CBBP":a.mean(),"mean_nonCBBP":b.mean(),
                      "mean_difference_CBBP_minus_nonCBBP":a.mean()-b.mean(),
                      "cohens_d":d,"welch_p":p,"mannwhitney_p":pu})
test_df=pd.DataFrame(tests)
if len(test_df):
    test_df["welch_p_BH"]=multipletests(test_df.welch_p,method="fdr_bh")[1]
display(test_df)
test_df.to_csv(TAB/"CBBP_nonCBBP_unadjusted_tests.csv",index=False)

In [ ]:
# Adjusted mixed-effects models: fixed effects plus animal random intercept.
# Simplify formula automatically if a categorical covariate has only one level.
covars=["Member","Breed","Sex","weight","Round","Description","Diurnality","feedtime"]
model_rows=[]
for y in primary_outcomes:
    use=[y,"AnimalID"]+[c for c in covars if c in df.columns]
    z=df[use].dropna().copy()
    terms=[]
    for c in covars:
        if c in z and z[c].nunique()>1:
            terms.append(f"C({c})" if z[c].dtype=="object" or c in ["Member","Breed","Sex","Round","Description","Diurnality","feedtime"] else c)
    if z.AnimalID.nunique()<10 or len(z)<30: continue
    formula=f"{y} ~ "+" + ".join(terms or ["1"])
    try:
        fit=mixedlm(formula,z,groups=z["AnimalID"]).fit(reml=False,method="lbfgs",disp=False)
        for term in fit.params.index:
            if term.startswith("C(Member)") or term=="Intercept":
                model_rows.append({"outcome":y,"term":term,"estimate":fit.params[term],
                                   "se":fit.bse[term],"p_value":fit.pvalues[term],
                                   "ci_low":fit.conf_int().loc[term,0],"ci_high":fit.conf_int().loc[term,1],
                                   "n":len(z),"n_animals":z.AnimalID.nunique(),"formula":formula})
        print(y, "converged:",fit.converged)
    except Exception as e:
        print("Mixed model failed for",y,":",str(e)[:250])
mixed_results=pd.DataFrame(model_rows)
display(mixed_results)
mixed_results.to_csv(TAB/"CBBP_nonCBBP_adjusted_mixed_models.csv",index=False)

## 6. Natural methane emission patterns

Assess variation by monitoring round, log day, diurnality, feed-time category, breed, sex and animal group. These are observational associations, not causal effects. Plot individual animal trajectories where repeated records exist and summarise by relevant strata.

In [ ]:
pattern_vars=["Round","log_day","Diurnality","feedtime","Breed","Sex","Description","Member"]
pattern_outcomes=[c for c in ["mean_methane","AUC_per_sec","eruct_per_min","n_peaks"] if c in df]
for g in pattern_vars:
    for y in pattern_outcomes:
        z=df[[g,y]].dropna()
        if z[g].nunique()<2: continue
        summary=z.groupby(g)[y].agg(["count","mean","median","std"]).sort_values("mean")
        summary.to_csv(TAB/f"pattern_{g}_{y}.csv")
        fig,ax=plt.subplots(figsize=(9,4))
        order=summary.index.tolist()
        sns.boxplot(data=z,x=g,y=y,order=order,ax=ax)
        ax.tick_params(axis="x",rotation=35)
        ax.set_title(f"{y} by {g}")
        fig.tight_layout(); fig.savefig(FIG/f"pattern_{g}_{y}.png",dpi=300); plt.show()

In [ ]:
# Animal-level trajectories over calendar date; only animals with >=2 observations.
trajectory_outcome="mean_methane"
repeated=df.groupby("AnimalID").filter(lambda x:len(x)>=2)
fig,ax=plt.subplots(figsize=(11,5))
for aid,g in repeated.groupby("AnimalID"):
    g=g.sort_values("file_date")
    ax.plot(g["file_date"],g[trajectory_outcome],alpha=.12,linewidth=.8)
daily=df.groupby("file_date")[trajectory_outcome].agg(["mean","sem","count"]).reset_index()
ax.errorbar(daily.file_date,daily["mean"],yerr=1.96*daily["sem"],fmt="o-",linewidth=2,capsize=2,label="Daily mean ± 95% approx. CI")
ax.set_title("Session-level methane trajectories and daily mean")
ax.set_xlabel("Measurement date"); ax.set_ylabel(trajectory_outcome); ax.legend()
fig.tight_layout(); fig.savefig(FIG/"animal_trajectories_and_daily_mean.png",dpi=300); plt.show()

## 7. High versus low methane emitter classification

A data-derived median split is convenient but biologically arbitrary. The primary classification below uses the animal-level median of a selected phenotype as a transparent exploratory threshold. A sensitivity analysis uses lower/upper tertiles and excludes the middle tertile. A validated biological threshold should replace these definitions if available.

To prevent leakage, the split is grouped by animal. Predictors exclude methane-derived variables and the target phenotype itself. Do not include post-outcome or target-derived features. Performance is reported using balanced accuracy, F1, ROC-AUC and average precision; compare against a majority-class baseline.

In [ ]:
CLASS_TARGET="mean_methane"  # Change to a justified primary phenotype, e.g. AUC_per_sec
# Animal-level phenotype median threshold, then map labels to sessions.
animal_target=df.groupby("AnimalID")[CLASS_TARGET].median().dropna()
threshold=float(animal_target.median())
df["emitter_class"]=np.where(df[CLASS_TARGET]>=threshold,"High","Low")
print("Session-level threshold:",threshold)
print(df["emitter_class"].value_counts(dropna=False))

# Sensitivity cohort: animal-level bottom and top tertiles; middle tertile omitted.
qlo,qhi=animal_target.quantile([1/3,2/3])
animal_class=pd.Series(index=animal_target.index,dtype="object")
animal_class[animal_target<=qlo]="Low"
animal_class[animal_target>=qhi]="High"
df["emitter_tertile_class"]=df.AnimalID.map(animal_class)
print("Tertile thresholds:",qlo,qhi)

In [ ]:
# Group-aware ML classification. Predictors are metadata/management variables only.
predictors=[c for c in ["Sex","Breed","weight","Round","log_day","Description","Member",
                        "Diurnality","feedtime","recording_duration","full_duration_sec"] if c in df.columns]
model_df=df.dropna(subset=["AnimalID","emitter_class"]).copy()
X=model_df[predictors]
y=(model_df["emitter_class"]=="High").astype(int)
groups=model_df["AnimalID"]
cat=[c for c in predictors if X[c].dtype=="object" or str(X[c].dtype)=="string"]
num=[c for c in predictors if c not in cat]
pre=ColumnTransformer([
    ("num",Pipeline([("impute",SimpleImputer(strategy="median")),("scale",StandardScaler())]),num),
    ("cat",Pipeline([("impute",SimpleImputer(strategy="most_frequent")),
                     ("onehot",OneHotEncoder(handle_unknown="ignore"))]),cat)
])
classifiers={
    "Dummy":DummyClassifier(strategy="prior"),
    "LogisticRegression":LogisticRegression(max_iter=2000,class_weight="balanced",random_state=SEED),
    "RandomForest":RandomForestClassifier(n_estimators=500,min_samples_leaf=3,class_weight="balanced",random_state=SEED,n_jobs=-1),
    "ExtraTrees":ExtraTreesClassifier(n_estimators=500,min_samples_leaf=3,class_weight="balanced",random_state=SEED,n_jobs=-1),
    "SVM_RBF":SVC(C=1.0,kernel="rbf",probability=True,class_weight="balanced",random_state=SEED)
}
n_groups=groups.nunique()
n_splits=min(5,n_groups)
cv=GroupKFold(n_splits=n_splits)
classification_rows=[]
for name,est in classifiers.items():
    pipe=Pipeline([("preprocess",pre),("model",est)])
    fold_metrics=[]
    for tr,te in cv.split(X,y,groups):
        pipe.fit(X.iloc[tr],y.iloc[tr])
        pred=pipe.predict(X.iloc[te])
        prob=pipe.predict_proba(X.iloc[te])[:,1] if hasattr(pipe[-1],"predict_proba") else pred
        fold_metrics.append({
            "accuracy":accuracy_score(y.iloc[te],pred),
            "balanced_accuracy":balanced_accuracy_score(y.iloc[te],pred),
            "precision":precision_score(y.iloc[te],pred,zero_division=0),
            "recall":recall_score(y.iloc[te],pred,zero_division=0),
            "f1":f1_score(y.iloc[te],pred,zero_division=0),
            "roc_auc":roc_auc_score(y.iloc[te],prob) if len(np.unique(y.iloc[te]))==2 else np.nan,
            "average_precision":average_precision_score(y.iloc[te],prob)
        })
    tmp=pd.DataFrame(fold_metrics)
    classification_rows.append({"model":name,**{f"{c}_mean":tmp[c].mean() for c in tmp},
                                **{f"{c}_sd":tmp[c].std() for c in tmp}})
class_results=pd.DataFrame(classification_rows).sort_values("balanced_accuracy_mean",ascending=False)
display(class_results)
class_results.to_csv(TAB/"high_low_classifier_groupCV.csv",index=False)

## 8. Predictability of continuous methane phenotypes

Predict candidate continuous outcomes from metadata and management covariates. GroupKFold ensures that all records from a given animal remain in one fold. Report R², RMSE and MAE. Negative test R² is possible and indicates performance worse than predicting the test-set mean.

This is predictive association, not causal inference. The covariates include measurement duration; run a sensitivity model excluding duration variables if duration is potentially part of the outcome construction.

In [ ]:
reg_predictors=[c for c in ["Sex","Breed","weight","Round","log_day","Description","Member",
                            "Diurnality","feedtime","recording_duration","full_duration_sec"] if c in df.columns]
reg_cat=[c for c in reg_predictors if df[c].dtype=="object" or str(df[c].dtype)=="string"]
reg_num=[c for c in reg_predictors if c not in reg_cat]
reg_pre=ColumnTransformer([
    ("num",Pipeline([("impute",SimpleImputer(strategy="median")),("scale",StandardScaler())]),reg_num),
    ("cat",Pipeline([("impute",SimpleImputer(strategy="most_frequent")),
                     ("onehot",OneHotEncoder(handle_unknown="ignore"))]),reg_cat)
], sparse_threshold=0.0)
regressors={
    "MeanBaseline":DummyRegressor(strategy="mean"),
    "Ridge":Ridge(alpha=1.0),
    "RandomForest":RandomForestRegressor(n_estimators=500,min_samples_leaf=3,max_features=.8,random_state=SEED,n_jobs=-1),
    "ExtraTrees":ExtraTreesRegressor(n_estimators=500,min_samples_leaf=3,max_features=.8,random_state=SEED,n_jobs=-1),
    "SVR_RBF":SVR(C=10,epsilon=.1)
}
reg_rows=[]
for outcome in ["mean_methane","AUC_per_sec","eruct_per_min","mean_resp"]:
    if outcome not in df: continue
    z=df.dropna(subset=[outcome,"AnimalID"]).copy()
    Xz=z[reg_predictors]; yz=z[outcome]; gz=z.AnimalID
    cvz=GroupKFold(n_splits=min(5,gz.nunique()))
    for name,est in regressors.items():
        pipe=Pipeline([("preprocess",reg_pre),("model",est)])
        fold=[]
        for tr,te in cvz.split(Xz,yz,gz):
            pipe.fit(Xz.iloc[tr],yz.iloc[tr]); pr=pipe.predict(Xz.iloc[te])
            fold.append({"r2":r2_score(yz.iloc[te],pr),
                         "rmse":mean_squared_error(yz.iloc[te],pr)**.5,
                         "mae":mean_absolute_error(yz.iloc[te],pr)})
        tmp=pd.DataFrame(fold)
        reg_rows.append({"outcome":outcome,"model":name,
                         **{f"{c}_mean":tmp[c].mean() for c in tmp},
                         **{f"{c}_sd":tmp[c].std() for c in tmp}})
reg_results=pd.DataFrame(reg_rows)
display(reg_results.sort_values(["outcome","rmse_mean"]))
reg_results.to_csv(TAB/"continuous_prediction_groupCV.csv",index=False)

## 9. SVM and feature importance

The SVM models above provide nonlinear classification/regression comparisons. Use permutation importance on a held-out animal-group test set for a selected outcome; importance is predictive contribution under this validation design, not a causal effect. Correlated predictors can share or mask importance.

In [ ]:
# Held-out grouped split and ExtraTrees permutation importance for continuous phenotype.
yname="mean_methane"
z=df.dropna(subset=[yname,"AnimalID"]).copy()
Xz=z[reg_predictors]; yz=z[yname]; gz=z.AnimalID
gss=GroupShuffleSplit(n_splits=1,test_size=.25,random_state=SEED)
tr,te=next(gss.split(Xz,yz,gz))
best_pipe=Pipeline([("preprocess",reg_pre),
                    ("model",ExtraTreesRegressor(n_estimators=600,min_samples_leaf=3,
                        random_state=SEED,n_jobs=-1))])
best_pipe.fit(Xz.iloc[tr],yz.iloc[tr])
pred=best_pipe.predict(Xz.iloc[te])
print("Held-out animals:",gz.iloc[te].nunique(),"R2:",r2_score(yz.iloc[te],pred),
      "RMSE:",mean_squared_error(yz.iloc[te],pred)**.5,"MAE:",mean_absolute_error(yz.iloc[te],pred))
perm=permutation_importance(best_pipe,Xz.iloc[te],yz.iloc[te],n_repeats=20,
                            random_state=SEED,scoring="neg_mean_absolute_error",n_jobs=-1)
imp=pd.DataFrame({"feature":reg_predictors,"importance_mean":perm.importances_mean,
                  "importance_sd":perm.importances_std}).sort_values("importance_mean",ascending=False)
display(imp)
imp.to_csv(TAB/"permutation_importance_mean_methane.csv",index=False)
joblib.dump(best_pipe,MOD/"extratrees_mean_methane.joblib")
fig,ax=plt.subplots(figsize=(8,5))
sns.barplot(data=imp.head(12),x="importance_mean",y="feature",ax=ax)
ax.set_title("Permutation importance: mean methane")
fig.tight_layout(); fig.savefig(FIG/"permutation_importance_mean_methane.png",dpi=300); plt.show()

## 10. ARIMA and Prophet: exploratory aggregate time-series analysis

The CSV does not contain timestamped methane observations within each 3–5 minute monitoring session. It does contain calendar dates and session summaries. This section therefore models the *daily mean of session-level methane phenotype* as an exploratory series. Days with no monitoring are not assumed to have zero emissions. A daily aggregate series may be sparse and short; forecasts should be labelled exploratory and not used as animal-level predictions.

The time series must be inspected for gaps, irregular sampling, outliers and changes in monitoring composition. Do not interpret an aggregate temporal trend as a biological trend without controlling for which animals were measured on each date.

In [ ]:
daily_ts=(df.dropna(subset=["file_date","mean_methane"])
          .groupby("file_date")["mean_methane"].mean().sort_index())
print("Observed dates:",len(daily_ts),"from",daily_ts.index.min(),"to",daily_ts.index.max())
display(daily_ts.describe().to_frame("daily_mean_methane"))
fig,ax=plt.subplots(figsize=(12,4))
daily_ts.plot(ax=ax,marker="o",linewidth=1)
ax.set_title("Daily mean of session-level mean methane")
ax.set_ylabel("mean_methane"); fig.tight_layout()
fig.savefig(FIG/"daily_mean_methane_timeseries.png",dpi=300); plt.show()

In [ ]:
# ARIMA: chronological holdout. Require enough observed dates; no automatic zero filling.
from statsmodels.tsa.arima.model import ARIMA
arima_result=None
if len(daily_ts)>=20:
    split=max(10,int(len(daily_ts)*.8))
    train_ts=daily_ts.iloc[:split]; test_ts=daily_ts.iloc[split:]
    candidates=[]
    for order in [(1,0,0),(0,1,1),(1,1,0),(1,1,1),(2,1,1)]:
        try:
            fit=ARIMA(train_ts,order=order).fit()
            fc=fit.forecast(steps=len(test_ts))
            candidates.append({"order":order,"aic":fit.aic,
                "mae":mean_absolute_error(test_ts,fc),
                "rmse":mean_squared_error(test_ts,fc)**.5})
        except Exception as e:
            print("ARIMA failed",order,str(e)[:120])
    arima_table=pd.DataFrame(candidates).sort_values("aic")
    display(arima_table)
    arima_table.to_csv(TAB/"ARIMA_candidate_comparison.csv",index=False)
    if len(arima_table):
        order=tuple(arima_table.iloc[0]["order"])
        arima_result=ARIMA(train_ts,order=order).fit()
        forecast=arima_result.forecast(steps=len(test_ts))
        print("Selected order:",order,"test MAE:",mean_absolute_error(test_ts,forecast))
        fig,ax=plt.subplots(figsize=(10,4))
        train_ts.plot(ax=ax,label="Train"); test_ts.plot(ax=ax,label="Observed test")
        forecast.plot(ax=ax,label="ARIMA forecast")
        ax.legend(); ax.set_title("Exploratory ARIMA forecast")
        fig.tight_layout(); fig.savefig(FIG/"ARIMA_forecast.png",dpi=300); plt.show()
else:
    print("Insufficient observed dates for a meaningful ARIMA holdout (need at least 20).")

In [ ]:
# Prophet is optional. Fit only if installed and there are enough dates.
try:
    from prophet import Prophet
    if len(daily_ts)>=30:
        prop_df=daily_ts.rename("y").reset_index()
        prop_df.columns=["ds","y"]
        split=max(20,int(len(prop_df)*.8))
        p_train=prop_df.iloc[:split]; p_test=prop_df.iloc[split:]
        prophet_model=Prophet(yearly_seasonality=False,weekly_seasonality=True,
                              daily_seasonality=False,interval_width=.95)
        prophet_model.fit(p_train)
        p_future=p_test[["ds"]].copy()
        p_forecast=prophet_model.predict(p_future)
        print("Prophet test MAE:",mean_absolute_error(p_test.y,p_forecast.yhat),
              "RMSE:",mean_squared_error(p_test.y,p_forecast.yhat)**.5)
        p_forecast[["ds","yhat","yhat_lower","yhat_upper"]].to_csv(TAB/"prophet_forecast.csv",index=False)
        fig=prophet_model.plot(p_forecast); fig.savefig(FIG/"prophet_forecast.png",dpi=300); plt.show()
    else:
        print("Prophet skipped: fewer than 30 observed dates.")
except ImportError:
    print("Prophet not installed; install with %pip install prophet to run this optional section.")

## 11. LSTM: optional exploratory daily-series model

An LSTM requires a sufficiently long, regular sequence. The observed daily series is likely sparse and irregular. The following code resamples to weekly means, leaves missing weeks missing, and only interpolates short gaps for an explicitly exploratory univariate model. This can smooth away real variation and should not be treated as confirmatory evidence. It is skipped if the resulting series is too short.

For a definitive LSTM analysis, obtain the raw time-stamped sensor signal and use animal/session-aware sequence construction, chronological validation and an appropriate baseline.

In [ ]:
try:
    import tensorflow as tf
    from sklearn.preprocessing import MinMaxScaler
    weekly=daily_ts.resample("W").mean()
    # Interpolate only for this exploratory demonstration; record the fraction filled.
    weekly_i=weekly.interpolate(limit=2,limit_direction="both")
    print("Weekly points:",len(weekly_i),"interpolated fraction:",weekly_i.isna().sum()/max(1,len(weekly_i)))
    if len(weekly_i.dropna())>=40:
        vals=weekly_i.dropna().values.reshape(-1,1)
        scaler=MinMaxScaler()
        scaled=scaler.fit_transform(vals)
        lookback=4
        Xseq=[]; yseq=[]
        for i in range(lookback,len(scaled)):
            Xseq.append(scaled[i-lookback:i,0]); yseq.append(scaled[i,0])
        Xseq=np.array(Xseq).reshape(-1,lookback,1); yseq=np.array(yseq)
        cut=max(1,int(len(Xseq)*.8))
        Xtr,Xte=Xseq[:cut],Xseq[cut:]; ytr,yte=yseq[:cut],yseq[cut:]
        tf.keras.utils.set_random_seed(SEED)
        model=tf.keras.Sequential([tf.keras.layers.Input((lookback,1)),
            tf.keras.layers.LSTM(24),tf.keras.layers.Dense(1)])
        model.compile(optimizer="adam",loss="mse")
        model.fit(Xtr,ytr,epochs=60,batch_size=8,verbose=0,
                  validation_split=.2 if len(Xtr)>=10 else 0)
        pr=scaler.inverse_transform(model.predict(Xte,verbose=0))
        obs=scaler.inverse_transform(yte.reshape(-1,1))
        print("LSTM exploratory MAE:",mean_absolute_error(obs,pr),
              "RMSE:",mean_squared_error(obs,pr)**.5)
        model.save(MOD/"exploratory_weekly_lstm.keras")
        pd.DataFrame({"observed":obs.ravel(),"predicted":pr.ravel()}).to_csv(TAB/"LSTM_weekly_predictions.csv",index=False)
    else:
        print("LSTM skipped: fewer than 40 usable weekly observations.")
except ImportError:
    print("TensorFlow not installed; optional LSTM section skipped.")

## 12. Conventional statistical model comparison and diagnostics

A mixed-effects model is generally more appropriate than ordinary least squares when repeated measurements are nested within animals. The code below fits a focused model for mean methane. Check residual distributions, heteroscedasticity, convergence and influential observations. If assumptions are poor, consider a transformed outcome, robust standard errors, bootstrap confidence intervals or a distribution appropriate to the outcome.

In [ ]:
diag_outcome="mean_methane"
diag_covars=["Member","Breed","Sex","weight","Round","log_day","Diurnality","feedtime"]
z=df[[diag_outcome,"AnimalID"]+diag_covars].dropna().copy()
formula=diag_outcome+" ~ "+" + ".join(
    [f"C({c})" if z[c].dtype=="object" else c for c in diag_covars if z[c].nunique()>1])
try:
    diag_fit=mixedlm(formula,z,groups=z.AnimalID).fit(reml=False,method="lbfgs",disp=False)
    print(diag_fit.summary())
    resid=diag_fit.resid; fitted=diag_fit.fittedvalues
    fig,axs=plt.subplots(1,2,figsize=(11,4))
    sns.scatterplot(x=fitted,y=resid,ax=axs[0],s=18)
    axs[0].axhline(0,linestyle="--"); axs[0].set_title("Residuals vs fitted")
    qqplot(resid,line="s",ax=axs[1]); axs[1].set_title("Normal Q-Q")
    fig.tight_layout(); fig.savefig(FIG/"mixed_model_diagnostics.png",dpi=300); plt.show()
    diag_fit.save(MOD/"mean_methane_mixedlm.pkl")
except Exception as e:
    print("Mixed model diagnostic fit failed:",repr(e))

## 13. Sensitivity analyses and robustness

Recommended minimum sensitivity checks:
- Repeat phenotype ranking with and without duration-normalised metrics.
- Compare median-split and tertile-based emitter classification.
- Refit prediction models without monitoring-duration variables.
- Stratify by breed and round where sample sizes permit.
- Exclude flagged records only under a documented, pre-specified QC rule.
- Evaluate performance on a later monitoring round if the same animals and measurement design allow a meaningful temporal validation.

In [ ]:
# Round-specific descriptive phenotype summaries.
round_summary=df.groupby("Round")[primary_outcomes].agg(["count","mean","std","median"])
display(round_summary)
round_summary.to_csv(TAB/"round_specific_phenotypes.csv")

# Compare model predictors without duration features.
no_duration=[c for c in reg_predictors if c not in ["recording_duration","full_duration_sec"]]
print("Full predictor set:",reg_predictors)
print("Duration-excluded predictor set:",no_duration)

# Check overlap of animals across rounds; informs paired/repeated design.
round_presence=df.groupby("AnimalID")["Round"].nunique().value_counts().sort_index()
print("Number of rounds represented per animal:")
display(round_presence.rename_axis("n_rounds").to_frame("n_animals"))

## 14. Publication-ready outputs and reporting checklist

The notebook exports descriptive tables, QC flags, phenotype correlations, ICC estimates, group comparisons, mixed-model estimates, prediction metrics, feature importance, time-series forecasts (when feasible), figures and fitted model files.

Before manuscript submission:
1. Confirm sensor calibration, units, signal processing and AUC definitions with the instrument protocol.
2. Define the biological estimand: concentration proxy, event rate, integrated exposure, or calibrated methane flux.
3. Pre-specify the primary phenotype and primary CBBP contrast.
4. State animal/session counts, repeated-measure structure, missingness and exclusions.
5. Report effect estimates with 95% confidence intervals and exact p-values; adjust for multiple testing where appropriate.
6. Describe grouped validation and ensure no animal leakage between folds.
7. Report class thresholds and class prevalence; do not present a median split as a validated biological threshold.
8. Present ARIMA/Prophet/LSTM as exploratory unless a sufficiently long, regularly sampled and biologically appropriate time series is available.
9. Include model diagnostics, uncertainty and external/temporal validation where possible.
10. Do not claim methane yield in g/day without a calibration/flux conversion.

### Export cleaned analysis dataset

In [ ]:
# Save a clean working copy including derived variables and class labels.
df.to_csv(OUT/"cleaned_methane_phenotype_dataset.csv",index=False)
print("Analysis completed. Outputs written to:",OUT.resolve())
print("Tables:",len(list(TAB.glob("*.csv"))),"Figures:",len(list(FIG.glob("*.png"))),
      "Models:",len(list(MOD.glob("*"))))

## 15. Suggested manuscript structure

**Title concept:** *Repeatability, predictive performance and management-associated variation in sheep methane emission phenotypes under community-based breeding and non-breeding systems.*

- **Introduction:** rationale for low-cost methane phenotyping in small ruminants; need to identify repeatable and predictive proxy phenotypes.
- **Materials and methods:** study population, monitoring protocol, variable definitions, QC, phenotype construction, repeatability, group comparisons, prediction/classification, temporal analysis and validation.
- **Results:** cohort description; phenotype distributions and intercorrelations; repeatability; CBBP contrasts; emission patterns; grouped-CV prediction; high/low classification; exploratory temporal models.
- **Discussion:** biological interpretation, implications for selection and breeding, measurement duration, repeatability, limitations of concentration proxies and the need for calibrated methane flux.
- **Conclusion:** identify the most defensible proxy phenotype only after considering repeatability, duration robustness and biological validation.

**Interpretive guardrail:** this notebook computes analyses from the supplied dataset; it does not pre-fill or fabricate results. Numerical findings in a manuscript must be taken from the executed outputs and checked against model diagnostics and the study protocol.

# 16. Formal benchmark: conventional statistical methods versus multiple ML algorithms

This section is the main model-comparison framework. It separates **explanatory/inferential modelling** from **out-of-animal prediction**:

- **Conventional methods:** ordinary least squares (OLS), multiple linear regression, generalized estimating equations (GEE), and linear mixed-effects models (LMM). These provide interpretable coefficients, uncertainty and adjustment for repeated measurements.
- **Machine learning:** Elastic Net, Ridge, K-nearest neighbours, support vector machines, Random Forest, Extra Trees, Gradient Boosting and HistGradientBoosting. Optional XGBoost/CatBoost are included when installed.
- **Validation:** grouped cross-validation by `AnimalID`, so records from one animal cannot occur in both training and test folds.
- **Regression metrics:** RMSE, MAE and R².
- **Classification metrics:** balanced accuracy, F1, ROC-AUC and average precision.

Do not rank models by a single metric alone. For skewed phenotypes, MAE and RMSE provide complementary error summaries. Negative test R² is valid and indicates poor generalisation.

**Important distinction:** Mixed-effects models and GEE are primarily used for population inference and repeated-measure structure. Their coefficient estimates and confidence intervals answer different questions from ML prediction scores. Compare predictive performance only where the validation target and prediction setting are equivalent.

In [ ]:
from sklearn.base import clone
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import make_scorer
from statsmodels.formula.api import gee
from statsmodels.genmod.cov_struct import Exchangeable
from statsmodels.genmod.families import Gaussian, Binomial

# A consistent predictor set is used across all algorithms.
BENCHMARK_FEATURES = [c for c in [
    "Sex","Breed","weight","Round","log_day","Description","Member",
    "Diurnality","feedtime","recording_duration","full_duration_sec"
] if c in df.columns]
BENCH_CAT = [c for c in BENCHMARK_FEATURES
             if df[c].dtype == "object" or str(df[c].dtype) == "string"]
BENCH_NUM = [c for c in BENCHMARK_FEATURES if c not in BENCH_CAT]

BENCH_PREPROCESS = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), BENCH_NUM),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), BENCH_CAT)
], sparse_threshold=0.0)

def regression_model_zoo():
    models = {
        "OLS_linear_regression": LinearRegression(),
        "Ridge": Ridge(alpha=1.0),
        "ElasticNet": ElasticNet(alpha=0.05, l1_ratio=0.5, max_iter=10000, random_state=SEED),
        "KNN": KNeighborsRegressor(n_neighbors=7, weights="distance"),
        "SVR_RBF": SVR(C=10, epsilon=0.1, kernel="rbf"),
        "RandomForest": RandomForestRegressor(n_estimators=500, min_samples_leaf=3,
                                               max_features=0.8, random_state=SEED, n_jobs=-1),
        "ExtraTrees": ExtraTreesRegressor(n_estimators=500, min_samples_leaf=3,
                                           max_features=0.8, random_state=SEED, n_jobs=-1),
        "GradientBoosting": GradientBoostingRegressor(n_estimators=250, learning_rate=0.04,
                                                       max_depth=2, random_state=SEED),
        "HistGradientBoosting": HistGradientBoostingRegressor(max_iter=250, learning_rate=0.05,
                                                               l2_regularization=1.0, random_state=SEED)
    }
    try:
        from xgboost import XGBRegressor
        models["XGBoost"] = XGBRegressor(n_estimators=400, max_depth=3, learning_rate=0.04,
                                          subsample=0.8, colsample_bytree=0.8,
                                          objective="reg:squarederror", random_state=SEED,
                                          n_jobs=-1)
    except ImportError:
        print("XGBoost not installed; skipped.")
    try:
        from catboost import CatBoostRegressor
        # One-hot preprocessing ensures compatibility with the common pipeline.
        models["CatBoost"] = CatBoostRegressor(iterations=400, depth=5, learning_rate=0.04,
                                                verbose=False, random_seed=SEED)
    except ImportError:
        print("CatBoost not installed; skipped.")
    return models

def grouped_regression_benchmark(data, outcome, features=BENCHMARK_FEATURES, n_splits=5):
    z=data.dropna(subset=[outcome, "AnimalID"]).copy()
    X=z[features].copy()
    y=z[outcome].astype(float)
    groups=z["AnimalID"]
    cv=GroupKFold(n_splits=min(n_splits, groups.nunique()))
    fold_records=[]
    model_dict=regression_model_zoo()
    for model_name, estimator in model_dict.items():
        for fold_no,(itr,ite) in enumerate(cv.split(X,y,groups), start=1):
            pipe=Pipeline([("preprocess", clone(BENCH_PREPROCESS)),
                           ("model", clone(estimator))])
            pipe.fit(X.iloc[itr],y.iloc[itr])
            pred=pipe.predict(X.iloc[ite])
            fold_records.append({
                "outcome":outcome,"model":model_name,"fold":fold_no,
                "n_train":len(itr),"n_test":len(ite),
                "n_train_animals":groups.iloc[itr].nunique(),
                "n_test_animals":groups.iloc[ite].nunique(),
                "R2":r2_score(y.iloc[ite],pred),
                "RMSE":mean_squared_error(y.iloc[ite],pred)**0.5,
                "MAE":mean_absolute_error(y.iloc[ite],pred)
            })
    folds=pd.DataFrame(fold_records)
    summary=(folds.groupby(["outcome","model"])
             .agg(R2_mean=("R2","mean"),R2_sd=("R2","std"),
                  RMSE_mean=("RMSE","mean"),RMSE_sd=("RMSE","std"),
                  MAE_mean=("MAE","mean"),MAE_sd=("MAE","std"))
             .reset_index().sort_values("RMSE_mean"))
    return folds,summary

# Run across the key continuous phenotypes.
REGRESSION_BENCHMARK_OUTCOMES = [c for c in
    ["mean_methane","AUC_per_sec","AUC_peak_per_sec","eruct_per_min","mean_resp"]
    if c in df.columns]
all_reg_folds=[]
all_reg_summaries=[]
for target in REGRESSION_BENCHMARK_OUTCOMES:
    f,s=grouped_regression_benchmark(df,target)
    all_reg_folds.append(f); all_reg_summaries.append(s)
reg_benchmark_folds=pd.concat(all_reg_folds,ignore_index=True)
reg_benchmark_summary=pd.concat(all_reg_summaries,ignore_index=True)
display(reg_benchmark_summary)
reg_benchmark_folds.to_csv(TAB/"all_regression_model_fold_scores.csv",index=False)
reg_benchmark_summary.to_csv(TAB/"all_regression_model_comparison.csv",index=False)

In [ ]:
# Visual model comparison: lower RMSE/MAE is better; higher R2 is better.
for target in REGRESSION_BENCHMARK_OUTCOMES:
    plot_data=reg_benchmark_summary.query("outcome == @target").sort_values("RMSE_mean")
    fig,ax=plt.subplots(figsize=(9,max(4,.38*len(plot_data))))
    sns.barplot(data=plot_data,x="RMSE_mean",y="model",ax=ax)
    ax.set_title(f"Grouped-CV model comparison: {target} (RMSE)")
    ax.set_xlabel("Mean fold RMSE (lower is better)")
    fig.tight_layout()
    fig.savefig(FIG/f"model_comparison_RMSE_{target}.png",dpi=300)
    plt.show()

# Export one best model ranking per phenotype using RMSE, with MAE and R2 retained.
best_regression_models=(reg_benchmark_summary.sort_values(["outcome","RMSE_mean"])
                        .groupby("outcome",as_index=False).first())
display(best_regression_models)
best_regression_models.to_csv(TAB/"best_regression_model_by_phenotype.csv",index=False)

## 16.1 Conventional repeated-measures models: OLS, GEE and mixed-effects models

For each primary phenotype, fit:
1. OLS with the same fixed-effect covariates (baseline conventional model; standard errors do not account for within-animal clustering).
2. GEE with an exchangeable working correlation grouped by animal (population-average effects with clustered dependence).
3. Linear mixed-effects model with an animal random intercept (subject-specific variation and intraclass correlation).

The GEE and mixed model are more appropriate than naive OLS for repeated animal records. Convergence warnings and singular fits must be reported, not hidden.

In [ ]:
CONVENTIONAL_COVARS = [c for c in
    ["Member","Breed","Sex","weight","Round","log_day","Description","Diurnality","feedtime"]
    if c in df.columns]

def conventional_repeated_models(data, outcome):
    cols=[outcome,"AnimalID"]+CONVENTIONAL_COVARS
    z=data[cols].dropna().copy()
    terms=[]
    for c in CONVENTIONAL_COVARS:
        if z[c].nunique()>1:
            terms.append(f"C({c})" if (z[c].dtype=="object" or str(z[c].dtype)=="string") else c)
    formula=f"{outcome} ~ "+" + ".join(terms or ["1"])
    results=[]
    # OLS
    try:
        fit_ols=ols(formula,z).fit()
        for term in fit_ols.params.index:
            results.append({"outcome":outcome,"method":"OLS","term":term,
                "estimate":fit_ols.params[term],"se":fit_ols.bse[term],
                "p_value":fit_ols.pvalues[term],
                "ci_low":fit_ols.conf_int().loc[term,0],
                "ci_high":fit_ols.conf_int().loc[term,1],
                "n":len(z),"n_animals":z.AnimalID.nunique(),
                "AIC":fit_ols.aic,"BIC":fit_ols.bic})
    except Exception as e:
        print("OLS failed",outcome,repr(e))
    # GEE: population-average model, clustered by animal.
    try:
        fit_gee=gee(formula,groups="AnimalID",data=z,
                    cov_struct=Exchangeable(),family=Gaussian()).fit()
        for term in fit_gee.params.index:
            ci=fit_gee.conf_int().loc[term]
            results.append({"outcome":outcome,"method":"GEE_exchangeable","term":term,
                "estimate":fit_gee.params[term],"se":fit_gee.bse[term],
                "p_value":fit_gee.pvalues[term],"ci_low":ci.iloc[0],"ci_high":ci.iloc[1],
                "n":len(z),"n_animals":z.AnimalID.nunique(),
                "AIC":np.nan,"BIC":np.nan})
    except Exception as e:
        print("GEE failed",outcome,repr(e))
    # Random-intercept mixed model.
    try:
        fit_lmm=mixedlm(formula,z,groups=z["AnimalID"]).fit(reml=False,method="lbfgs",disp=False)
        for term in fit_lmm.fe_params.index:
            ci=fit_lmm.conf_int().loc[term]
            results.append({"outcome":outcome,"method":"LMM_random_intercept","term":term,
                "estimate":fit_lmm.fe_params[term],"se":fit_lmm.bse[term],
                "p_value":fit_lmm.pvalues[term],"ci_low":ci.iloc[0],"ci_high":ci.iloc[1],
                "n":len(z),"n_animals":z.AnimalID.nunique(),
                "AIC":fit_lmm.aic,"BIC":fit_lmm.bic})
    except Exception as e:
        print("LMM failed",outcome,repr(e))
    return pd.DataFrame(results)

conv_results=[]
for target in REGRESSION_BENCHMARK_OUTCOMES:
    conv_results.append(conventional_repeated_models(df,target))
conventional_model_results=pd.concat(conv_results,ignore_index=True)
display(conventional_model_results.head(40))
conventional_model_results.to_csv(TAB/"conventional_OLS_GEE_LMM_coefficients.csv",index=False)

## 16.2 Classification benchmark: conventional logistic regression versus ML

Use the same animal-level median definition of high/low emitter used earlier. The threshold is derived from animal-level median methane and then assigned to that animal's records. This avoids defining an animal's class differently across its repeated sessions. Classification remains exploratory until a biological threshold is available.

In [ ]:
def classification_model_zoo():
    models={
        "LogisticRegression":LogisticRegression(max_iter=3000,class_weight="balanced",random_state=SEED),
        "KNN":KNeighborsClassifier(n_neighbors=7,weights="distance"),
        "SVM_RBF":SVC(C=1.0,kernel="rbf",probability=True,class_weight="balanced",random_state=SEED),
        "RandomForest":RandomForestClassifier(n_estimators=500,min_samples_leaf=3,
                                               class_weight="balanced",random_state=SEED,n_jobs=-1),
        "ExtraTrees":ExtraTreesClassifier(n_estimators=500,min_samples_leaf=3,
                                           class_weight="balanced",random_state=SEED,n_jobs=-1),
        "GradientBoosting":GradientBoostingClassifier(n_estimators=250,learning_rate=.04,
                                                       max_depth=2,random_state=SEED),
        "HistGradientBoosting":HistGradientBoostingClassifier(max_iter=250,learning_rate=.05,
                                                               l2_regularization=1.0,random_state=SEED)
    }
    try:
        from xgboost import XGBClassifier
        models["XGBoost"]=XGBClassifier(n_estimators=400,max_depth=3,learning_rate=.04,
            subsample=.8,colsample_bytree=.8,eval_metric="logloss",random_state=SEED,n_jobs=-1)
    except ImportError:
        print("XGBoost not installed; classification skipped.")
    return models

# Build a single animal-level label based on its median target phenotype.
animal_med=df.groupby("AnimalID")[CLASS_TARGET].median().dropna()
animal_cut=animal_med.median()
animal_labels=(animal_med>=animal_cut).astype(int)
clz=df[df.AnimalID.isin(animal_labels.index)].copy()
clz["target_binary"]=clz.AnimalID.map(animal_labels).astype(int)
Xc=clz[BENCHMARK_FEATURES]
yc=clz["target_binary"]
gc=clz["AnimalID"]
cv_class=GroupKFold(n_splits=min(5,gc.nunique()))
classification_fold_rows=[]
for model_name,est in classification_model_zoo().items():
    for fold_no,(itr,ite) in enumerate(cv_class.split(Xc,yc,gc),start=1):
        pipe=Pipeline([("preprocess",clone(BENCH_PREPROCESS)),("model",clone(est))])
        pipe.fit(Xc.iloc[itr],yc.iloc[itr])
        pred=pipe.predict(Xc.iloc[ite])
        prob=pipe.predict_proba(Xc.iloc[ite])[:,1] if hasattr(pipe[-1],"predict_proba") else pred
        classification_fold_rows.append({
            "model":model_name,"fold":fold_no,
            "accuracy":accuracy_score(yc.iloc[ite],pred),
            "balanced_accuracy":balanced_accuracy_score(yc.iloc[ite],pred),
            "precision":precision_score(yc.iloc[ite],pred,zero_division=0),
            "recall":recall_score(yc.iloc[ite],pred,zero_division=0),
            "F1":f1_score(yc.iloc[ite],pred,zero_division=0),
            "ROC_AUC":roc_auc_score(yc.iloc[ite],prob) if yc.iloc[ite].nunique()==2 else np.nan,
            "average_precision":average_precision_score(yc.iloc[ite],prob)
        })
classification_benchmark_folds=pd.DataFrame(classification_fold_rows)
classification_benchmark_summary=(classification_benchmark_folds.groupby("model")
    .agg(**{f"{m}_mean":(m,"mean") for m in
        ["accuracy","balanced_accuracy","precision","recall","F1","ROC_AUC","average_precision"]},
         **{f"{m}_sd":(m,"std") for m in
        ["accuracy","balanced_accuracy","precision","recall","F1","ROC_AUC","average_precision"]})
    .reset_index().sort_values("balanced_accuracy_mean",ascending=False))
display(classification_benchmark_summary)
classification_benchmark_folds.to_csv(TAB/"classification_all_models_fold_scores.csv",index=False)
classification_benchmark_summary.to_csv(TAB/"classification_all_models_comparison.csv",index=False)

In [ ]:
fig,ax=plt.subplots(figsize=(9,5))
sns.barplot(data=classification_benchmark_summary.sort_values("balanced_accuracy_mean",ascending=False),
            x="balanced_accuracy_mean",y="model",ax=ax)
ax.set_xlim(0,1); ax.set_xlabel("Mean grouped-CV balanced accuracy")
ax.set_title("Conventional logistic regression versus ML classifiers")
fig.tight_layout(); fig.savefig(FIG/"classification_model_comparison.png",dpi=300); plt.show()

## 16.3 Model comparison: how to report the results

Use the exported comparison tables to report:
- The best-performing ML algorithm for each phenotype, based on grouped-CV RMSE/MAE and R².
- Whether the best ML method meaningfully improves on OLS/Ridge and the mean baseline.
- The variability of performance across folds.
- Whether logistic regression is competitive with nonlinear classifiers for high/low emitter classification.
- GEE/LMM coefficient estimates and confidence intervals for adjusted population differences, separately from prediction metrics.

Do not claim an ML model is superior to a mixed model solely because its R² is higher: the estimands differ. For a fair predictive comparison, compare models under the same grouped folds and prediction target. For inferential comparison, report fixed-effect estimates, uncertainty and diagnostics.

**Recommended manuscript table set**
1. Table 1: animal and monitoring-session characteristics.
2. Table 2: methane phenotype descriptive statistics and ICC.
3. Table 3: CBBP versus Non-CBBP effects from GEE/LMM (estimate, 95% CI, p-value).
4. Table 4: grouped-CV continuous prediction comparison (R², RMSE, MAE).
5. Table 5: grouped-CV high/low classification comparison (balanced accuracy, F1, ROC-AUC, average precision).
6. Figure: phenotype correlation heatmap; repeatability plot; model-performance comparison; feature-importance plot.

The best model is not selected by training fit. It is selected using out-of-animal validation, uncertainty and biological plausibility.

# 17. Expanded scientific analyses: unsupervised phenotyping, clustering and multivariate structure

This section adds discovery-oriented analyses beyond supervised prediction. The primary unsupervised unit is the **animal**, not the monitoring session: repeated records are first summarised within animal to prevent animals with more observations from dominating the clusters.

Clustering identifies statistical profiles in the observed phenotype space. It does not establish discrete biological types, genetic classes, or causal mechanisms. Cluster number and interpretation must be supported by internal validity, stability, biological coherence and replication in an independent cohort or monitoring round.

## 17.1 Construct animal-level phenotype profiles

Use robust animal-level medians for methane outcomes and means/medians for continuous covariates. Retain the number of records and within-animal variability as quality/context indicators, but do not automatically include these in the clustering feature matrix because they may encode sampling intensity rather than biology.

In [ ]:
# Aggregate repeated sessions to one row per animal.
animal_agg_spec = {
    "mean_methane":"median", "mean_resp":"median", "mean_eruct":"median",
    "max_resp":"median", "max_eruct":"median", "eruct_per_min":"median",
    "AUC_peak_per_sec":"median", "AUC_per_sec":"median",
    "AUC_peaks_3min":"median", "AUC_3min":"median",
    "weight":"median", "n_peaks":"median", "n_resp":"median", "n_eruct":"median",
    "full_duration_sec":"median", "recording_duration":"median",
    "log_day":"median"
}
animal_agg_spec={k:v for k,v in animal_agg_spec.items() if k in df.columns}
animal_level=df.groupby("AnimalID").agg(animal_agg_spec)
animal_level["n_sessions"]=df.groupby("AnimalID").size()
for c in ["Breed","Sex","Member","Round","Description","Dataset"]:
    if c in df:
        animal_level[c]=df.groupby("AnimalID")[c].agg(
            lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
animal_level.to_csv(TAB/"animal_level_phenotype_profiles.csv")
print("Animal-level profile:",animal_level.shape)
display(animal_level.head())

## 17.2 Correlation structure and multicollinearity

Strongly correlated phenotype variables can represent overlapping measurements of the same signal. Clustering on many redundant variables may overweight one physiological dimension. Inspect the correlation matrix and variance inflation factors, then use a biologically justified reduced feature set.

In [ ]:
cluster_candidates=[c for c in [
    "mean_methane","mean_resp","mean_eruct","max_resp","max_eruct",
    "eruct_per_min","AUC_peak_per_sec","AUC_per_sec"
] if c in animal_level.columns]
cluster_corr=animal_level[cluster_candidates].corr(method="spearman")
display(cluster_corr.round(2))
fig,ax=plt.subplots(figsize=(9,7))
sns.heatmap(cluster_corr,annot=True,fmt=".2f",cmap="vlag",center=0,ax=ax)
ax.set_title("Animal-level methane phenotype correlation structure")
fig.tight_layout(); fig.savefig(FIG/"animal_level_cluster_feature_correlations.png",dpi=300); plt.show()
cluster_corr.to_csv(TAB/"animal_level_cluster_feature_correlations.csv")

## 17.3 PCA: dimensionality reduction and phenotype axes

PCA is performed on standardised, median-imputed animal-level phenotype variables. Report variance explained and loadings. Principal components are statistical axes, not direct biological pathways. Interpret a component from the joint loading pattern, not the largest loading alone.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score, adjusted_rand_score
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture

pca_features=cluster_candidates.copy()
pca_frame=animal_level[pca_features].replace([np.inf,-np.inf],np.nan)
pca_imputer=SimpleImputer(strategy="median")
X_pca_raw=pca_imputer.fit_transform(pca_frame)
pca_scaler=StandardScaler()
X_pca_scaled=pca_scaler.fit_transform(X_pca_raw)
pca_model=PCA(random_state=SEED)
scores=pca_model.fit_transform(X_pca_scaled)
explained=pd.DataFrame({
    "PC":[f"PC{i+1}" for i in range(len(pca_model.explained_variance_ratio_))],
    "variance_explained":pca_model.explained_variance_ratio_,
    "cumulative_variance":np.cumsum(pca_model.explained_variance_ratio_)
})
loadings=pd.DataFrame(pca_model.components_.T,index=pca_features,
                      columns=explained.PC)
display(explained.head(10))
display(loadings.iloc[:,:min(5,loadings.shape[1])].round(3))
explained.to_csv(TAB/"PCA_variance_explained.csv",index=False)
loadings.to_csv(TAB/"PCA_loadings.csv")
fig,ax=plt.subplots(figsize=(7,4))
ax.plot(range(1,len(explained)+1),explained.cumulative_variance,marker="o")
ax.axhline(.80,ls="--",color="grey"); ax.axhline(.90,ls=":",color="grey")
ax.set(xlabel="Number of principal components",ylabel="Cumulative variance explained",
       title="PCA cumulative variance")
fig.tight_layout(); fig.savefig(FIG/"PCA_scree_cumulative.png",dpi=300); plt.show()
if scores.shape[1]>=2:
    pca_plot=pd.DataFrame({"PC1":scores[:,0],"PC2":scores[:,1]},index=animal_level.index)
    fig,ax=plt.subplots(figsize=(7,6))
    sns.scatterplot(data=pca_plot,x="PC1",y="PC2",s=35,alpha=.75,ax=ax)
    ax.set_title("Animal-level phenotype ordination (PCA)")
    fig.tight_layout(); fig.savefig(FIG/"PCA_animal_ordination.png",dpi=300); plt.show()

## 17.4 Cluster-number selection: K-means, hierarchical clustering and Gaussian mixtures

Evaluate candidate cluster counts (k = 2–7) using silhouette, Davies–Bouldin and Calinski–Harabasz indices. These indices have different directions: higher silhouette and Calinski–Harabasz are generally preferred; lower Davies–Bouldin is preferred. They are diagnostics, not automatic proof of the true number of biological groups.

Use a robustly scaled phenotype matrix to reduce sensitivity to extreme values. The cluster feature set must be chosen before examining cluster outcomes where possible.

In [ ]:
cluster_imputer=SimpleImputer(strategy="median")
X_cluster_imp=cluster_imputer.fit_transform(animal_level[pca_features])
cluster_scaler=RobustScaler()
X_cluster=cluster_scaler.fit_transform(X_cluster_imp)
cluster_eval=[]
k_values=range(2,min(8,len(animal_level)-1))
for k in k_values:
    km=KMeans(n_clusters=k,n_init=50,random_state=SEED)
    lab=km.fit_predict(X_cluster)
    cluster_eval.append({"algorithm":"KMeans","k":k,
        "silhouette":silhouette_score(X_cluster,lab),
        "davies_bouldin":davies_bouldin_score(X_cluster,lab),
        "calinski_harabasz":calinski_harabasz_score(X_cluster,lab),
        "inertia":km.inertia_})
    ag=AgglomerativeClustering(n_clusters=k,linkage="ward")
    alab=ag.fit_predict(X_cluster)
    cluster_eval.append({"algorithm":"Ward_hierarchical","k":k,
        "silhouette":silhouette_score(X_cluster,alab),
        "davies_bouldin":davies_bouldin_score(X_cluster,alab),
        "calinski_harabasz":calinski_harabasz_score(X_cluster,alab),
        "inertia":np.nan})
    gm=GaussianMixture(n_components=k,covariance_type="full",n_init=10,
                       random_state=SEED,reg_covar=1e-5)
    glab=gm.fit_predict(X_cluster)
    if len(np.unique(glab))>1:
        cluster_eval.append({"algorithm":"GaussianMixture","k":k,
            "silhouette":silhouette_score(X_cluster,glab),
            "davies_bouldin":davies_bouldin_score(X_cluster,glab),
            "calinski_harabasz":calinski_harabasz_score(X_cluster,glab),
            "inertia":np.nan,"BIC":gm.bic(X_cluster),"AIC":gm.aic(X_cluster)})
cluster_eval_df=pd.DataFrame(cluster_eval)
display(cluster_eval_df)
cluster_eval_df.to_csv(TAB/"clustering_algorithm_k_selection.csv",index=False)

## 17.5 Cluster stability and final cluster solution

A cluster solution that changes substantially with random initialisation or resampling is not robust. The following assesses K-means stability across random seeds using adjusted Rand index (ARI). ARI near 1 indicates strong agreement; values near 0 indicate agreement comparable to chance.

Choose the final k only after jointly considering internal metrics, stability, minimum cluster size, interpretability and scientific rationale. The code uses the selected k below as a transparent provisional setting; change it after reviewing the diagnostics.

In [ ]:
# Provisional selection: update after reviewing the k-selection table.
SELECTED_K=2  # Provisional: k=2 had the highest silhouette in the initial k=2..7 screening; verify stability and biological coherence.
seed_labels=[]
for seed in range(20):
    labels=KMeans(n_clusters=SELECTED_K,n_init=1,random_state=seed).fit_predict(X_cluster)
    seed_labels.append(labels)
ari_values=[]
for i in range(len(seed_labels)):
    for j in range(i+1,len(seed_labels)):
        ari_values.append(adjusted_rand_score(seed_labels[i],seed_labels[j]))
print("K-means stability ARI mean:",np.mean(ari_values),
      "SD:",np.std(ari_values),"minimum:",np.min(ari_values))
final_kmeans=KMeans(n_clusters=SELECTED_K,n_init=100,random_state=SEED)
animal_level["methane_cluster"]=final_kmeans.fit_predict(X_cluster)+1
print(animal_level["methane_cluster"].value_counts().sort_index())
animal_level[["methane_cluster"]].to_csv(TAB/"animal_cluster_assignments.csv")

## 17.6 Cluster profiling and biological interpretation

Describe each cluster using the original measurement units, not z-scores alone. Report animal counts and percentages, median and interquartile range for each phenotype, and the distribution of breed, sex, CBBP membership and monitoring round. These metadata are used to characterise clusters after formation; they are not used to create the methane phenotype clusters in this analysis.

In [ ]:
cluster_profile=animal_level.groupby("methane_cluster")[pca_features].agg(["count","median","mean","std"])
display(cluster_profile.round(3))
cluster_profile.to_csv(TAB/"methane_cluster_phenotype_profiles.csv")
cluster_counts=animal_level["methane_cluster"].value_counts().sort_index()
cluster_counts.to_csv(TAB/"methane_cluster_sizes.csv",header=["n_animals"])
for meta in ["Breed","Sex","Member","Round","Description"]:
    if meta in animal_level:
        tab=pd.crosstab(animal_level["methane_cluster"],animal_level[meta],normalize="index")*100
        display(tab.round(1))
        tab.to_csv(TAB/f"cluster_composition_{meta}.csv")
# Standardised cluster-centroid heatmap for pattern comparison.
centroids=pd.DataFrame(cluster_scaler.inverse_transform(final_kmeans.cluster_centers_),
                       columns=pca_features,index=[f"Cluster {i}" for i in range(1,SELECTED_K+1)])
fig,ax=plt.subplots(figsize=(10,4))
sns.heatmap(centroids,annot=True,fmt=".2f",cmap="vlag",center=centroids.values.mean(),ax=ax)
ax.set_title("Cluster centroids in original phenotype units")
fig.tight_layout(); fig.savefig(FIG/"methane_cluster_centroids.png",dpi=300); plt.show()

## 17.7 Unsupervised algorithm sensitivity: hierarchical clustering and DBSCAN

Hierarchical clustering provides a dendrogram-based view of nested similarity. DBSCAN can identify dense groups and outliers without pre-specifying k, but its results are sensitive to `eps` and `min_samples`; it may classify many animals as noise in moderate-dimensional phenotype space. Treat DBSCAN as a sensitivity analysis, not the default final solution.

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram
Z=linkage(X_cluster,method="ward")
fig,ax=plt.subplots(figsize=(13,5))
dendrogram(Z,truncate_mode="lastp",p=30,leaf_rotation=45,leaf_font_size=8,ax=ax)
ax.set_title("Ward hierarchical clustering dendrogram (truncated)")
ax.set_ylabel("Ward linkage distance")
fig.tight_layout(); fig.savefig(FIG/"hierarchical_dendrogram.png",dpi=300); plt.show()

dbscan_rows=[]
for eps in [.5,.75,1.0,1.25,1.5,2.0]:
    labels=DBSCAN(eps=eps,min_samples=8).fit_predict(X_cluster)
    ncl=len(set(labels)-{-1})
    noise=np.mean(labels==-1)
    sil=np.nan
    mask=labels!=-1
    if ncl>=2 and mask.sum()>ncl:
        sil=silhouette_score(X_cluster[mask],labels[mask])
    dbscan_rows.append({"eps":eps,"min_samples":8,"n_clusters_excluding_noise":ncl,
                        "noise_fraction":noise,"silhouette_non_noise":sil})
dbscan_sensitivity=pd.DataFrame(dbscan_rows)
display(dbscan_sensitivity)
dbscan_sensitivity.to_csv(TAB/"DBSCAN_sensitivity.csv",index=False)

## 17.8 Statistical association of clusters with flock and animal characteristics

After deriving clusters from methane phenotypes alone, test whether cluster membership is associated with CBBP status, breed, sex, round and other metadata. Use chi-square tests with expected-cell diagnostics and Cramér's V for categorical associations. For continuous traits not used in clustering (e.g., weight if omitted from the cluster features), use Kruskal–Wallis tests and report effect sizes. Correct families of exploratory tests using Benjamini–Hochberg FDR.

In [ ]:
def cramers_v(table):
    chi2=stats.chi2_contingency(table,correction=False)[0]
    n=table.to_numpy().sum()
    r,k=table.shape
    phi2=chi2/n if n else np.nan
    return np.sqrt(phi2/max(1,min(k-1,r-1))) if n else np.nan

cluster_assoc=[]
for meta in ["Member","Breed","Sex","Round","Description"]:
    if meta in animal_level:
        tab=pd.crosstab(animal_level["methane_cluster"],animal_level[meta])
        if tab.shape[0]>1 and tab.shape[1]>1:
            chi2,p,dof,expected=stats.chi2_contingency(tab)
            cluster_assoc.append({"variable":meta,"test":"Chi-square",
                "statistic":chi2,"df":dof,"p_value":p,"cramers_v":cramers_v(tab),
                "min_expected_cell":expected.min()})
for var in ["weight","n_sessions"]:
    if var in animal_level:
        groups=[g[var].dropna().values for _,g in animal_level.groupby("methane_cluster")]
        if len(groups)>=2 and all(len(g)>0 for g in groups):
            h,p=stats.kruskal(*groups)
            n=sum(map(len,groups)); k=len(groups)
            eps2=max(0,(h-k+1)/(n-k)) if n>k else np.nan
            cluster_assoc.append({"variable":var,"test":"Kruskal-Wallis",
                "statistic":h,"df":k-1,"p_value":p,"effect_size_epsilon_squared":eps2})
cluster_assoc_df=pd.DataFrame(cluster_assoc)
if len(cluster_assoc_df):
    cluster_assoc_df["p_value_FDR"]=multipletests(cluster_assoc_df.p_value,method="fdr_bh")[1]
display(cluster_assoc_df)
cluster_assoc_df.to_csv(TAB/"cluster_metadata_associations.csv",index=False)

## 17.9 Multivariate group separation: PERMANOVA-style permutation test

This animal-level permutation test evaluates whether multivariate phenotype profiles differ by a grouping variable (e.g., CBBP membership or breed), using Euclidean distances in the standardised phenotype space. It is a one-factor exploratory test. Check multivariate dispersion as well: significant PERMANOVA can reflect differences in within-group dispersion rather than group centroids. Do not interpret this as causal.

In [ ]:
from scipy.spatial.distance import pdist, squareform

def permanova_oneway(X, groups, n_perm=999, seed=42):
    X=np.asarray(X); groups=np.asarray(groups)
    keep=pd.notna(groups)
    X=X[keep]; groups=groups[keep]
    levels=np.unique(groups)
    n=len(groups); k=len(levels)
    if k<2 or n<=k: return {"pseudo_F":np.nan,"p_value":np.nan,"n":n,"groups":k}
    D=squareform(pdist(X,metric="euclidean"))
    D2=D**2
    ss_total=np.sum(D2)/(2*n)
    ss_within=0.0
    for lev in levels:
        idx=np.where(groups==lev)[0]
        if len(idx)>1:
            ss_within += np.sum(D2[np.ix_(idx,idx)])/(2*len(idx))
    ss_between=ss_total-ss_within
    pseudo_F=(ss_between/(k-1))/(ss_within/(n-k)) if ss_within>0 else np.nan
    rng=np.random.default_rng(seed); exceed=0
    for _ in range(n_perm):
        perm=rng.permutation(groups)
        sw=0.0
        for lev in levels:
            idx=np.where(perm==lev)[0]
            if len(idx)>1: sw += np.sum(D2[np.ix_(idx,idx)])/(2*len(idx))
        sb=ss_total-sw
        f=(sb/(k-1))/(sw/(n-k)) if sw>0 else np.nan
        if f>=pseudo_F: exceed+=1
    return {"pseudo_F":pseudo_F,"p_value":(exceed+1)/(n_perm+1),"n":n,"groups":k}

permanova_results=[]
for factor in ["Member","Breed","Sex","Round"]:
    if factor in animal_level:
        valid=animal_level[factor].notna().values
        res=permanova_oneway(X_cluster[valid],animal_level.loc[valid,factor].values,
                             n_perm=999,seed=SEED)
        res["factor"]=factor
        permanova_results.append(res)
permanova_df=pd.DataFrame(permanova_results)
if len(permanova_df):
    permanova_df["p_value_FDR"]=multipletests(permanova_df.p_value,method="fdr_bh")[1]
display(permanova_df)
permanova_df.to_csv(TAB/"animal_level_PERMANOVA_exploratory.csv",index=False)

## 17.10 Longitudinal consistency between monitoring rounds

For animals observed in both rounds, compare within-animal phenotype values across R1 and R2. Use paired tests and Spearman rank correlation. This assesses longitudinal rank consistency across the year and may reflect biological change, management change, environment, measurement variation and changes in protocol; it is not pure instrument repeatability.

In [ ]:
longitudinal=[]
if "Round" in df:
    for outcome in ["mean_methane","AUC_per_sec","AUC_peak_per_sec","eruct_per_min"]:
        if outcome not in df: continue
        wide=df[df.Round.isin(["R1","R2"])].groupby(["AnimalID","Round"])[outcome].median().unstack()
        if {"R1","R2"}.issubset(wide.columns):
            paired=wide[["R1","R2"]].dropna()
            if len(paired)>=5:
                wil=stats.wilcoxon(paired.R1,paired.R2)
                rho,rp=stats.spearmanr(paired.R1,paired.R2)
                diff=paired.R2-paired.R1
                longitudinal.append({"outcome":outcome,"n_paired_animals":len(paired),
                    "R1_median":paired.R1.median(),"R2_median":paired.R2.median(),
                    "median_change_R2_minus_R1":diff.median(),"wilcoxon_p":wil.pvalue,
                    "spearman_rank_rho":rho,"rank_correlation_p":rp})
longitudinal_df=pd.DataFrame(longitudinal)
if len(longitudinal_df):
    longitudinal_df["wilcoxon_p_FDR"]=multipletests(longitudinal_df.wilcoxon_p,method="fdr_bh")[1]
display(longitudinal_df)
longitudinal_df.to_csv(TAB/"longitudinal_round_comparison.csv",index=False)

## 17.11 Animal-cluster bootstrap confidence intervals for CBBP contrasts

Use animal-level summaries and resample animals within CBBP membership groups to estimate confidence intervals for group mean differences. This prevents repeated sessions from artificially increasing the effective sample size. These bootstrap intervals are complementary to the adjusted GEE/LMM models and do not replace covariate adjustment.

In [ ]:
def stratified_animal_bootstrap_diff(data, outcome, group_col="Member",
                                     group_a="CBBP", group_b="NON-CBBP",
                                     n_boot=3000, seed=42):
    z=data[["AnimalID",group_col,outcome]].dropna().copy()
    a=z[z[group_col]==group_a].groupby("AnimalID")[outcome].median().dropna().values
    b=z[z[group_col]==group_b].groupby("AnimalID")[outcome].median().dropna().values
    if len(a)<2 or len(b)<2: return None
    observed=np.mean(a)-np.mean(b)
    rng=np.random.default_rng(seed)
    boots=np.empty(n_boot)
    for i in range(n_boot):
        aa=rng.choice(a,size=len(a),replace=True)
        bb=rng.choice(b,size=len(b),replace=True)
        boots[i]=aa.mean()-bb.mean()
    return {"outcome":outcome,"n_animals_CBBP":len(a),"n_animals_nonCBBP":len(b),
            "mean_difference":observed,"bootstrap_CI_low":np.quantile(boots,.025),
            "bootstrap_CI_high":np.quantile(boots,.975),"bootstrap_SE":boots.std(ddof=1)}

boot_results=[]
for outcome in primary_outcomes:
    res=stratified_animal_bootstrap_diff(df,outcome,n_boot=3000,seed=SEED)
    if res: boot_results.append(res)
bootstrap_CBBP=pd.DataFrame(boot_results)
display(bootstrap_CBBP)
bootstrap_CBBP.to_csv(TAB/"animal_bootstrap_CBBP_differences.csv",index=False)

## 17.12 Explainable ML: SHAP (optional) and permutation importance

Permutation importance is already calculated in the supervised modelling section. SHAP can provide local and global model explanations, but explanations are conditional on the fitted model and observed feature distribution. Correlated predictors can make SHAP attributions unstable. Install SHAP if required and interpret results as model explanations, not causal effects.

In [ ]:
try:
    import shap
    # Use a tree model fitted to the selected phenotype, with the preprocessing pipeline.
    # SHAP is computed on a manageable sample to control runtime.
    shap_model=best_pipe.named_steps["model"]
    Xte_trans=best_pipe.named_steps["preprocess"].transform(Xz.iloc[te])
    feature_names=best_pipe.named_steps["preprocess"].get_feature_names_out()
    sample_n=min(200,Xte_trans.shape[0])
    explainer=shap.TreeExplainer(shap_model)
    shap_values=explainer.shap_values(Xte_trans[:sample_n])
    shap.summary_plot(shap_values,Xte_trans[:sample_n],
                      feature_names=feature_names,show=False)
    plt.tight_layout(); plt.savefig(FIG/"SHAP_summary_mean_methane.png",dpi=300,bbox_inches="tight")
    plt.show()
except ImportError:
    print("SHAP is optional. Install with %pip install shap to generate SHAP explanations.")
except Exception as e:
    print("SHAP analysis could not be completed:",repr(e))

## 17.13 Scientific interpretation rules for clustering and multivariate analyses

- Cluster on animal-level methane phenotypes; use metadata only for post hoc characterisation.
- Report the number of animals per cluster and any small/unstable cluster.
- Present internal validity indices and seed/resampling stability.
- Avoid naming clusters "high emitters" or "low emitters" until their profiles demonstrate this consistently across multiple appropriate phenotype measures.
- Validate cluster structure in a held-out monitoring round or independent flock if sample size and design permit.
- Do not use cluster labels as ground truth for supervised classification on the same dataset.
- PCA and clustering are exploratory and require external biological validation.
- PERMANOVA results should be interpreted alongside dispersion diagnostics.
- Apply FDR correction across families of exploratory hypothesis tests.

# 18. Final reproducibility and manuscript export manifest

This section records key analysis choices and creates a manifest of generated outputs. Before publication, execute the notebook from top to bottom in a clean environment and archive the environment specification, input data version, notebook, outputs and random seed.

In [ ]:
analysis_manifest={
    "input_file":str(DATA_PATH),
    "input_shape":list(df_raw.shape),
    "unique_animals":int(df_raw["AnimalID"].nunique()),
    "random_seed":SEED,
    "animal_grouped_validation":True,
    "primary_regression_outcomes":REGRESSION_BENCHMARK_OUTCOMES,
    "cluster_features":pca_features,
    "selected_cluster_k_provisional":SELECTED_K,
    "classification_target":CLASS_TARGET,
    "classification_threshold_animal_median":float(animal_cut),
    "python_version":sys.version,
    "outputs":{
        "tables":len(list(TAB.glob("*.csv"))),
        "figures":len(list(FIG.glob("*.png"))),
        "models":len(list(MOD.glob("*")))
    }
}
with open(OUT/"analysis_manifest.json","w") as f:
    json.dump(analysis_manifest,f,indent=2,default=str)
print(json.dumps(analysis_manifest,indent=2,default=str))

## 17.14 Multivariate dispersion diagnostic for PERMANOVA

PERMANOVA can be significant because groups differ in their multivariate centroids, their within-group dispersion, or both. As a complementary diagnostic, calculate each animal's Euclidean distance to its group centroid in the standardised phenotype space and compare these distances across groups. This is an approximate dispersion check; for formal distance-based multivariate dispersion inference, use a validated PERMDISP implementation (e.g., `vegan::betadisper` in R) and report it alongside PERMANOVA.

In [ ]:
dispersion_rows=[]
for factor in ["Member","Breed","Sex","Round"]:
    if factor not in animal_level: continue
    valid=animal_level[factor].notna().values
    Xf=X_cluster[valid]
    gf=animal_level.loc[valid,factor].astype(str).values
    dist_to_centroid=np.full(len(gf),np.nan)
    for level in np.unique(gf):
        idx=np.where(gf==level)[0]
        centroid=Xf[idx].mean(axis=0)
        dist_to_centroid[idx]=np.linalg.norm(Xf[idx]-centroid,axis=1)
    group_dist=[dist_to_centroid[gf==level] for level in np.unique(gf)]
    if len(group_dist)>=2:
        lev_stat,lev_p=stats.levene(*group_dist,center="median")
        kw_stat,kw_p=stats.kruskal(*group_dist)
        dispersion_rows.append({"factor":factor,"levene_statistic":lev_stat,
                                "levene_p":lev_p,"kruskal_statistic":kw_stat,
                                "kruskal_p":kw_p})
dispersion_df=pd.DataFrame(dispersion_rows)
if len(dispersion_df):
    dispersion_df["levene_p_FDR"]=multipletests(dispersion_df.levene_p,method="fdr_bh")[1]
display(dispersion_df)
dispersion_df.to_csv(TAB/"multivariate_dispersion_diagnostic.csv",index=False)

## 17.15 Nested grouped cross-validation for tuned ML models

The initial benchmark uses pre-specified model settings and grouped cross-validation. For a more rigorous final comparison, hyperparameter tuning must occur *inside* each training fold, while the outer folds estimate generalisation. The following nested grouped CV is provided for a selected phenotype and three representative algorithms. It is computationally more intensive than the baseline benchmark but avoids using the outer test animals to select hyperparameters.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, loguniform

RUN_NESTED_CV=True
NESTED_TARGET="mean_methane"
if RUN_NESTED_CV:
    zn=df.dropna(subset=[NESTED_TARGET,"AnimalID"]).copy()
    Xn=zn[BENCHMARK_FEATURES]
    yn=zn[NESTED_TARGET].astype(float)
    gn=zn["AnimalID"]
    outer=GroupKFold(n_splits=min(3,gn.nunique()))
    nested_specs={
        "ExtraTrees":{
            "estimator":ExtraTreesRegressor(random_state=SEED,n_jobs=-1),
            "params":{
                "model__n_estimators":[300,500],
                "model__max_depth":[None,8,15],
                "model__min_samples_leaf":[2,3,5,8],
                "model__max_features":[0.6,0.8,1.0]
            }},
        "RandomForest":{
            "estimator":RandomForestRegressor(random_state=SEED,n_jobs=-1),
            "params":{
                "model__n_estimators":[300,500],
                "model__max_depth":[None,8,15],
                "model__min_samples_leaf":[2,3,5,8],
                "model__max_features":[0.6,0.8,1.0]
            }},
        "SVR":{
            "estimator":SVR(kernel="rbf"),
            "params":{
                "model__C":[0.1,1,10,50,100],
                "model__epsilon":[0.05,0.1,0.2,0.5],
                "model__gamma":["scale","auto"]
            }}
    }
    nested_rows=[]
    for model_name,spec in nested_specs.items():
        for fold,(itr,ite) in enumerate(outer.split(Xn,yn,gn),start=1):
            pipe=Pipeline([("preprocess",clone(BENCH_PREPROCESS)),
                           ("model",clone(spec["estimator"]))])
            inner=GroupKFold(n_splits=min(3,gn.iloc[itr].nunique()))
            search=RandomizedSearchCV(
                pipe,param_distributions=spec["params"],n_iter=8,
                scoring="neg_root_mean_squared_error",cv=inner,
                random_state=SEED,n_jobs=-1,refit=True)
            search.fit(Xn.iloc[itr],yn.iloc[itr],groups=gn.iloc[itr])
            pred=search.best_estimator_.predict(Xn.iloc[ite])
            nested_rows.append({
                "model":model_name,"outer_fold":fold,
                "R2":r2_score(yn.iloc[ite],pred),
                "RMSE":mean_squared_error(yn.iloc[ite],pred)**0.5,
                "MAE":mean_absolute_error(yn.iloc[ite],pred),
                "best_params":str(search.best_params_)
            })
    nested_results=pd.DataFrame(nested_rows)
    display(nested_results)
    nested_results.to_csv(TAB/"nested_grouped_CV_tuned_ML.csv",index=False)
    nested_summary=(nested_results.groupby("model")
        .agg(R2_mean=("R2","mean"),R2_sd=("R2","std"),
             RMSE_mean=("RMSE","mean"),RMSE_sd=("RMSE","std"),
             MAE_mean=("MAE","mean"),MAE_sd=("MAE","std")).reset_index()
        .sort_values("RMSE_mean"))
    display(nested_summary)
    nested_summary.to_csv(TAB/"nested_grouped_CV_tuned_ML_summary.csv",index=False)
else:
    print("Nested grouped CV disabled by RUN_NESTED_CV=False.")

## 17.16 Classification probability calibration

For a high/low emitter classifier, discrimination (ROC-AUC) is not the same as calibration. If the model will be used to prioritise animals for further measurement or selection, assess whether predicted probabilities correspond to observed event frequencies. Report Brier score and calibration curves in addition to discrimination metrics. Calibration must be assessed on held-out animals only.

In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss
# Use the fitted best classifier only after choosing it using grouped CV.
# This is a single grouped holdout illustration; repeat or bootstrap for uncertainty.
class_holdout=GroupShuffleSplit(n_splits=1,test_size=.25,random_state=SEED)
itr,ite=next(class_holdout.split(Xc,yc,gc))
cal_model=Pipeline([
    ("preprocess",clone(BENCH_PREPROCESS)),
    ("model",LogisticRegression(max_iter=3000,class_weight="balanced",random_state=SEED))
])
cal_model.fit(Xc.iloc[itr],yc.iloc[itr])
cal_prob=cal_model.predict_proba(Xc.iloc[ite])[:,1]
cal_true=yc.iloc[ite].values
brier=brier_score_loss(cal_true,cal_prob)
frac_pos,mean_pred=calibration_curve(cal_true,cal_prob,n_bins=8,strategy="quantile")
print("Held-out animal-group Brier score:",brier)
fig,ax=plt.subplots(figsize=(6,6))
ax.plot(mean_pred,frac_pos,marker="o",label="Logistic regression")
ax.plot([0,1],[0,1],ls="--",color="grey",label="Perfect calibration")
ax.set(xlabel="Mean predicted probability",ylabel="Observed fraction high emitters",
       title="High-emitter probability calibration")
ax.legend(); fig.tight_layout()
fig.savefig(FIG/"high_emitter_calibration_curve.png",dpi=300); plt.show()


# 19. Additional scientific analyses: variance components and animal-level reliability

This module extends the existing ICC analysis by partitioning variance attributable to animal identity and available design factors. It fits exploratory random-intercept models for each phenotype. The random-effect estimates are conditional on the available sampling design; they are not estimates of genetic heritability.


In [ ]:
variance_component_rows = []
vc_phenotypes = [c for c in [
    "mean_methane", "mean_resp", "mean_eruct", "max_resp", "max_eruct",
    "eruct_per_min", "AUC_peak_per_sec", "AUC_per_sec"
] if c in df.columns]

for outcome in vc_phenotypes:
    cols = [outcome, "AnimalID"] + [c for c in ["Round", "log_day", "Breed", "Sex", "Member"] if c in df.columns]
    tmp = df[cols].replace([np.inf, -np.inf], np.nan).dropna(subset=[outcome, "AnimalID"]).copy()
    if tmp["AnimalID"].nunique() < 5 or len(tmp) < 15:
        continue
    try:
        # Random intercept for animal; design variables are fixed effects where estimable.
        fixed = [c for c in ["Round", "log_day", "Breed", "Sex", "Member"] if c in tmp.columns and tmp[c].nunique() > 1]
        formula = f'Q("{outcome}") ~ ' + (" + ".join([f'C(Q("{c}"))' if tmp[c].dtype == "object" else f'Q("{c}")' for c in fixed]) if fixed else "1")
        fit = mixedlm(formula, tmp, groups=tmp["AnimalID"]).fit(reml=True, method="lbfgs", disp=False)
        animal_var = float(fit.cov_re.iloc[0, 0])
        resid_var = float(fit.scale)
        total = animal_var + resid_var
        variance_component_rows.append({
            "phenotype": outcome, "n_records": len(tmp), "n_animals": tmp.AnimalID.nunique(),
            "animal_variance": animal_var, "residual_variance": resid_var,
            "animal_fraction_of_modelled_variance": animal_var / total if total > 0 else np.nan,
            "converged": bool(getattr(fit, "converged", True))
        })
    except Exception as e:
        variance_component_rows.append({"phenotype": outcome, "n_records": len(tmp),
            "n_animals": tmp.AnimalID.nunique(), "error": str(e)[:250]})

variance_components_df = pd.DataFrame(variance_component_rows)
display(variance_components_df)
variance_components_df.to_csv(TAB / "variance_components_animal_random_intercept.csv", index=False)


# 20. Allometric scaling and body-size adjustment

Absolute methane-related measurements may scale with body size. This module compares raw phenotypes with body-weight-adjusted and metabolic-body-weight-scaled indices, and fits log-log allometric models. Ratios are exploratory and should not replace covariate-adjusted models because ratio standardisation can introduce artefacts.


In [ ]:
allometry_rows = []
allometry_data = df.copy()
if "weight" in allometry_data.columns:
    allometry_data["metabolic_bw"] = pd.to_numeric(allometry_data["weight"], errors="coerce").clip(lower=1) ** 0.75
    for outcome in vc_phenotypes:
        x = allometry_data[["weight", "metabolic_bw", outcome]].apply(pd.to_numeric, errors="coerce").replace([np.inf,-np.inf], np.nan).dropna()
        x = x[(x.weight > 0) & (x[outcome] > 0)]
        if len(x) < 15:
            continue
        try:
            fit = ols(f'np.log(Q("{outcome}")) ~ np.log(weight)', data=x).fit(cov_type="HC3")
            allometry_rows.append({
                "phenotype": outcome, "n": len(x), "log_log_slope": fit.params.get("np.log(weight)", np.nan),
                "slope_ci_low": fit.conf_int().loc["np.log(weight)", 0],
                "slope_ci_high": fit.conf_int().loc["np.log(weight)", 1],
                "p_value": fit.pvalues.get("np.log(weight)", np.nan), "r_squared": fit.rsquared
            })
            allometry_data[f"{outcome}_per_kg075"] = pd.to_numeric(allometry_data[outcome], errors="coerce") / allometry_data["metabolic_bw"]
            allometry_data[f"{outcome}_per_kg"] = pd.to_numeric(allometry_data[outcome], errors="coerce") / pd.to_numeric(allometry_data["weight"], errors="coerce")
        except Exception as e:
            print("Allometry failed for", outcome, e)
allometry_df = pd.DataFrame(allometry_rows)
display(allometry_df)
allometry_df.to_csv(TAB / "allometric_log_log_models.csv", index=False)
scaled_cols = [c for c in allometry_data if c.endswith("_per_kg075") or c.endswith("_per_kg")]
if scaled_cols:
    allometry_data[["AnimalID"] + scaled_cols].to_csv(TAB / "body_size_scaled_phenotypes.csv", index=False)


# 21. Nonlinear and distribution-sensitive association models: GAM-like splines and quantile regression

Restricted cubic-like spline terms (via patsy B-splines) assess nonlinear associations with weight and recording time. Quantile regression evaluates whether covariate associations differ across the methane phenotype distribution. These are association models, not causal effects.


In [ ]:
from patsy import bs
nonlinear_rows, quantile_rows = [], []
for outcome in [c for c in ["mean_methane", "AUC_per_sec", "eruct_per_min"] if c in df.columns]:
    covars = [c for c in ["weight", "recording_duration"] if c in df.columns]
    use = [outcome] + covars + [c for c in ["Sex", "Breed", "Member", "Round"] if c in df.columns]
    tmp = df[use].replace([np.inf,-np.inf],np.nan).dropna(subset=[outcome]).copy()
    if len(tmp) < 40:
        continue
    for c in covars:
        tmp[c] = pd.to_numeric(tmp[c], errors="coerce")
    tmp = tmp.dropna(subset=covars)
    if len(tmp) < 40:
        continue
    try:
        terms = [f'bs(Q("{c}"), df=4, degree=3)' for c in covars if tmp[c].nunique() >= 6]
        categorical = [f'C(Q("{c}"))' for c in ["Sex","Breed","Member","Round"] if c in tmp and tmp[c].nunique() > 1]
        formula = f'Q("{outcome}") ~ ' + " + ".join(terms + categorical) if terms or categorical else f'Q("{outcome}") ~ 1'
        fit = ols(formula, tmp).fit(cov_type="HC3")
        nonlinear_rows.append({"phenotype": outcome, "n": len(tmp), "formula": formula,
                               "r_squared": fit.rsquared, "adjusted_r_squared": fit.rsquared_adj,
                               "aic": fit.aic})
    except Exception as e:
        nonlinear_rows.append({"phenotype": outcome, "error": str(e)[:250]})
    if "weight" in tmp and tmp["weight"].nunique() >= 8:
        for tau in [0.10, 0.25, 0.50, 0.75, 0.90]:
            try:
                qfit = sm.QuantReg(tmp[outcome], sm.add_constant(tmp[["weight"]])).fit(q=tau, max_iter=5000)
                quantile_rows.append({"phenotype": outcome, "quantile": tau,
                                      "weight_coefficient": qfit.params.get("weight", np.nan),
                                      "weight_p_value": qfit.pvalues.get("weight", np.nan),
                                      "n": len(tmp)})
            except Exception as e:
                print("Quantile model warning:", outcome, tau, str(e)[:150])
nonlinear_df, quantile_df = pd.DataFrame(nonlinear_rows), pd.DataFrame(quantile_rows)
display(nonlinear_df); display(quantile_df)
nonlinear_df.to_csv(TAB / "nonlinear_spline_associations.csv", index=False)
quantile_df.to_csv(TAB / "quantile_regression_weight_associations.csv", index=False)


# 22. Longitudinal mixed-effects analysis across rounds and monitoring days

This module fits animal-random-intercept models with round, monitoring day and selected covariates. It also harmonises round labels and estimates paired within-animal changes. A two-round design does not support complex trajectory claims; interpret round contrasts as temporal differences under the observed protocol.


In [ ]:
longitudinal_rows = []
if "AnimalID" in df.columns:
    for outcome in [c for c in ["mean_methane","AUC_per_sec","eruct_per_min","mean_eruct"] if c in df.columns]:
        cols = [outcome, "AnimalID"] + [c for c in ["Round","log_day","weight","Sex","Breed","Member","Diurnality","feedtime"] if c in df.columns]
        tmp = df[cols].replace([np.inf,-np.inf],np.nan).dropna(subset=[outcome,"AnimalID"]).copy()
        if tmp.AnimalID.nunique() < 5 or len(tmp) < 20:
            continue
        fixed = [c for c in ["Round","log_day","weight","Sex","Breed","Member","Diurnality","feedtime"] if c in tmp and tmp[c].nunique() > 1]
        terms = [f'C(Q("{c}"))' if tmp[c].dtype == "object" else f'Q("{c}")' for c in fixed]
        formula = f'Q("{outcome}") ~ ' + " + ".join(terms) if terms else f'Q("{outcome}") ~ 1'
        try:
            m = mixedlm(formula, tmp, groups=tmp["AnimalID"]).fit(reml=True, method="lbfgs", disp=False)
            for term, coef in m.params.items():
                longitudinal_rows.append({"phenotype":outcome,"term":term,"estimate":coef,
                    "se":m.bse.get(term,np.nan),"p_value":m.pvalues.get(term,np.nan),
                    "n":len(tmp),"n_animals":tmp.AnimalID.nunique(),"converged":getattr(m,"converged",True)})
        except Exception as e:
            longitudinal_rows.append({"phenotype":outcome,"error":str(e)[:250]})
longitudinal_mixed_df = pd.DataFrame(longitudinal_rows)
display(longitudinal_mixed_df.head(30))
longitudinal_mixed_df.to_csv(TAB / "longitudinal_mixed_effects_fixed_effects.csv", index=False)

# Paired round comparison after robustly mapping common round encodings.
paired_round_rows = []
if "Round" in df.columns:
    rr = df.copy()
    rr["_round_norm"] = rr["Round"].astype(str).str.lower().str.replace("round","r",regex=False).str.replace(" ","",regex=False)
    rr["_round_norm"] = rr["_round_norm"].replace({"1":"r1","2":"r2","2024":"r1","2025":"r2"})
    for outcome in [c for c in ["mean_methane","AUC_per_sec","eruct_per_min","mean_eruct"] if c in rr]:
        wide = rr.groupby(["AnimalID","_round_norm"])[outcome].median().unstack()
        if {"r1","r2"}.issubset(wide.columns):
            pair = wide[["r1","r2"]].dropna()
            if len(pair) >= 5:
                w,p = stats.wilcoxon(pair.r1,pair.r2)
                rho,rp = stats.spearmanr(pair.r1,pair.r2)
                paired_round_rows.append({"phenotype":outcome,"n_paired_animals":len(pair),
                    "median_R1":pair.r1.median(),"median_R2":pair.r2.median(),
                    "median_change_R2_minus_R1":(pair.r2-pair.r1).median(),
                    "wilcoxon_statistic":w,"wilcoxon_p":p,"spearman_rho":rho,"spearman_p":rp})
paired_round_df = pd.DataFrame(paired_round_rows)
display(paired_round_df)
paired_round_df.to_csv(TAB / "paired_round_comparisons_harmonised.csv", index=False)


# 23. CBBP/non-CBBP equivalence and effect-size analysis

A nonsignificant difference is not evidence of equivalence. This section calculates animal-level standardised differences and performs two one-sided tests (TOST) using a user-defined smallest effect size of interest. The default margin of ±0.20 SD is a conventional small-effect threshold and must be justified biologically before publication.


In [ ]:
TOST_MARGIN = 0.20  # SD units; prespecify/justify before interpreting
equivalence_rows = []
if "Member" in animal_level.columns:
    for outcome in [c for c in cluster_candidates if c in animal_level.columns]:
        tmp = animal_level[[outcome,"Member"]].dropna()
        labels = tmp["Member"].astype(str).str.upper()
        a = tmp.loc[labels.str.contains("CBBP"), outcome].astype(float).values
        b = tmp.loc[labels.str.contains("NON"), outcome].astype(float).values
        if len(a) >= 5 and len(b) >= 5:
            pooled = np.sqrt(((len(a)-1)*np.var(a,ddof=1)+(len(b)-1)*np.var(b,ddof=1))/(len(a)+len(b)-2))
            d = (np.mean(a)-np.mean(b))/pooled if pooled > 0 else np.nan
            # TOST on standardised difference, using Welch standard error.
            se = np.sqrt(np.var(a,ddof=1)/len(a)+np.var(b,ddof=1)/len(b))
            diff = np.mean(a)-np.mean(b)
            df_w = (np.var(a,ddof=1)/len(a)+np.var(b,ddof=1)/len(b))**2 / (
                (np.var(a,ddof=1)/len(a))**2/(len(a)-1) + (np.var(b,ddof=1)/len(b))**2/(len(b)-1))
            margin_raw = TOST_MARGIN * pooled
            t_lower = (diff + margin_raw)/se
            t_upper = (diff - margin_raw)/se
            p_lower = 1-stats.t.cdf(t_lower,df_w)
            p_upper = stats.t.cdf(t_upper,df_w)
            equivalence_rows.append({"phenotype":outcome,"n_CBBP":len(a),"n_non_CBBP":len(b),
                "standardised_mean_difference":d,"raw_mean_difference_CBBP_minus_non":diff,
                "tost_margin_sd":TOST_MARGIN,"tost_p_lower":p_lower,"tost_p_upper":p_upper,
                "equivalent_at_alpha_0_05":bool(max(p_lower,p_upper)<0.05)})
equivalence_df = pd.DataFrame(equivalence_rows)
display(equivalence_df)
equivalence_df.to_csv(TAB / "CBBP_nonCBBP_equivalence_TOST.csv", index=False)


# 24. Consensus clustering and resampling-based cluster robustness

This module compares K-means, Ward hierarchical clustering and Gaussian mixtures on the animal-level phenotype matrix. A consensus co-assignment matrix is estimated by repeated subsampling. This is a robustness diagnostic, not proof that discrete biological classes exist.


In [ ]:
from sklearn.metrics import adjusted_rand_score
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.mixture import GaussianMixture

CONSENSUS_K = SELECTED_K if "SELECTED_K" in globals() else 2
Xc = np.asarray(X_cluster)
n_anim = Xc.shape[0]
rng = np.random.default_rng(SEED)
coassign = np.zeros((n_anim,n_anim),dtype=float)
coobserved = np.zeros((n_anim,n_anim),dtype=float)
algo_labels = {}
for algo in ["KMeans","Ward","GMM"]:
    try:
        if algo == "KMeans":
            lab = KMeans(n_clusters=CONSENSUS_K,n_init=50,random_state=SEED).fit_predict(Xc)
        elif algo == "Ward":
            lab = AgglomerativeClustering(n_clusters=CONSENSUS_K,linkage="ward").fit_predict(Xc)
        else:
            lab = GaussianMixture(n_components=CONSENSUS_K,n_init=10,random_state=SEED,reg_covar=1e-5).fit_predict(Xc)
        algo_labels[algo] = lab
    except Exception as e:
        print(algo, "failed:", e)

for b in range(100):
    idx = np.sort(rng.choice(n_anim,size=max(CONSENSUS_K+2,int(0.80*n_anim)),replace=False))
    lab = KMeans(n_clusters=CONSENSUS_K,n_init=20,random_state=SEED+b).fit_predict(Xc[idx])
    coobserved[np.ix_(idx,idx)] += 1
    same = (lab[:,None] == lab[None,:]).astype(float)
    coassign[np.ix_(idx,idx)] += same
consensus = np.divide(coassign,coobserved,out=np.full_like(coassign,np.nan),where=coobserved>0)
fig,ax=plt.subplots(figsize=(8,7))
sns.heatmap(consensus, cmap="viridis", vmin=0, vmax=1, cbar_kws={"label":"Co-clustering probability"}, ax=ax)
ax.set_title("Consensus co-clustering matrix (subsampled K-means)")
fig.tight_layout(); fig.savefig(FIG/"consensus_clustering_matrix.png",dpi=300); plt.show()
pd.DataFrame(consensus,index=animal_level.index,columns=animal_level.index).to_csv(TAB/"consensus_clustering_matrix.csv")
agreement_rows=[]
for a in algo_labels:
    for b in algo_labels:
        if a < b:
            agreement_rows.append({"algorithm_1":a,"algorithm_2":b,
                                   "adjusted_rand_index":adjusted_rand_score(algo_labels[a],algo_labels[b])})
agreement_df=pd.DataFrame(agreement_rows)
display(agreement_df)
agreement_df.to_csv(TAB/"clustering_algorithm_agreement.csv",index=False)


# 25. Longitudinal cluster transitions between rounds

Animals with measurements in both rounds are assigned round-specific clusters using a common scaling basis and fixed centroids derived from the pooled animal-level profiles. Transition counts and row percentages describe profile stability. Cluster numbers are arbitrary; interpret transitions by comparing cluster phenotype centroids.


In [ ]:
transition_outputs = []
if "Round" in df.columns and "AnimalID" in df.columns:
    rr = df.copy()
    rr["_round_norm"] = rr["Round"].astype(str).str.lower().str.replace("round","r",regex=False).str.replace(" ","",regex=False)
    rr["_round_norm"] = rr["_round_norm"].replace({"1":"r1","2":"r2","2024":"r1","2025":"r2"})
    round_agg = rr.groupby(["AnimalID","_round_norm"])[cluster_candidates].median().reset_index()
    for rnd in ["r1","r2"]:
        sub = round_agg[round_agg["_round_norm"]==rnd].set_index("AnimalID")
        vals = sub.reindex(animal_level.index)[cluster_candidates]
        arr = cluster_scaler.transform(cluster_imputer.transform(vals))
        animal_level[f"cluster_{rnd}"] = final_kmeans.predict(arr)+1
    both = animal_level.dropna(subset=["cluster_r1","cluster_r2"]).copy()
    if len(both):
        trans = pd.crosstab(both["cluster_r1"],both["cluster_r2"])
        trans_pct = pd.crosstab(both["cluster_r1"],both["cluster_r2"],normalize="index")*100
        display(trans); display(trans_pct.round(1))
        trans.to_csv(TAB/"cluster_transition_counts_R1_R2.csv")
        trans_pct.to_csv(TAB/"cluster_transition_row_percent_R1_R2.csv")
        ari = adjusted_rand_score(both.cluster_r1,both.cluster_r2)
        print("Round-to-round cluster assignment ARI:", round(ari,3), "| paired animals:",len(both))
        pd.DataFrame([{"n_paired_animals":len(both),"ARI_R1_R2":ari}]).to_csv(TAB/"cluster_transition_stability_summary.csv",index=False)
else:
    print("Round or AnimalID unavailable; cluster transition analysis skipped.")


# 26. Recording-duration sensitivity and minimum-duration assessment

Recording duration can affect event counts and integrated AUC phenotypes. This module examines duration associations, compares phenotype summaries across progressively stricter duration filters, and evaluates whether animal rankings are robust. Thresholds are empirical sensitivity cut-offs, not validated minimum recording standards.


In [ ]:
duration_sensitivity_rows=[]
duration_col = "recording_duration" if "recording_duration" in df.columns else ("full_duration_sec" if "full_duration_sec" in df.columns else None)
if duration_col:
    dur = pd.to_numeric(df[duration_col],errors="coerce")
    duration_thresholds = {"all_records": -np.inf,
        "above_P25": dur.quantile(.25), "above_median": dur.median(), "above_P75": dur.quantile(.75)}
    for outcome in [c for c in ["mean_methane","AUC_full","AUC_3min","AUC_per_sec","AUC_peak_per_sec","n_peaks"] if c in df.columns]:
        for label, threshold in duration_thresholds.items():
            z=df[[ "AnimalID", outcome, duration_col]].copy()
            z[outcome]=pd.to_numeric(z[outcome],errors="coerce")
            z[duration_col]=pd.to_numeric(z[duration_col],errors="coerce")
            z=z[z[duration_col]>=threshold].dropna(subset=["AnimalID",outcome])
            med=z.groupby("AnimalID")[outcome].median()
            duration_sensitivity_rows.append({"phenotype":outcome,"duration_filter":label,
                "duration_threshold":threshold if np.isfinite(threshold) else np.nan,
                "n_records":len(z),"n_animals":med.size,"mean_animal_median":med.mean(),
                "median_animal_median":med.median()})
duration_sensitivity_df=pd.DataFrame(duration_sensitivity_rows)
display(duration_sensitivity_df)
duration_sensitivity_df.to_csv(TAB/"recording_duration_sensitivity.csv",index=False)

if duration_col and "AnimalID" in df.columns:
    rank_rows=[]
    base=df.groupby("AnimalID")["mean_methane"].median() if "mean_methane" in df else None
    if base is not None:
        for label, threshold in duration_thresholds.items():
            z=df[pd.to_numeric(df[duration_col],errors="coerce")>=threshold]
            cur=z.groupby("AnimalID")["mean_methane"].median()
            common=base.index.intersection(cur.index)
            if len(common)>=5:
                rho,p=stats.spearmanr(base.loc[common],cur.loc[common])
                rank_rows.append({"duration_filter":label,"n_animals":len(common),"rank_spearman":rho,"p_value":p})
        duration_rank_df=pd.DataFrame(rank_rows)
        display(duration_rank_df)
        duration_rank_df.to_csv(TAB/"recording_duration_animal_rank_stability.csv",index=False)


# 27. Biological interaction analyses

This module tests a small, prespecified set of biologically motivated interactions. Models adjust for available covariates and use animal-clustered robust standard errors where possible. Interactions are exploratory and should be limited to hypotheses with adequate cell sizes.


In [ ]:
interaction_rows=[]
interaction_specs=[
    ("weight","Breed"),("weight","Sex"),("Member","Breed"),
    ("Round","Breed"),("Diurnality","feedtime")
]
for outcome in [c for c in ["mean_methane","AUC_per_sec","eruct_per_min"] if c in df.columns]:
    for a,b in interaction_specs:
        if a not in df.columns or b not in df.columns: continue
        needed=[outcome,a,b,"AnimalID"]+[c for c in ["Round","Sex","Breed","Member","weight","recording_duration"] if c in df.columns and c not in [a,b]]
        tmp=df[needed].replace([np.inf,-np.inf],np.nan).dropna(subset=[outcome,a,b]).copy()
        if len(tmp)<40 or tmp[a].nunique()<2 or tmp[b].nunique()<2: continue
        # Avoid unstable high-dimensional interactions.
        if tmp[a].nunique()*tmp[b].nunique()>30: continue
        terms=[f'C(Q("{a}"))' if tmp[a].dtype=="object" else f'Q("{a}")',
               f'C(Q("{b}"))' if tmp[b].dtype=="object" else f'Q("{b}")']
        interaction=f'{terms[0]}:{terms[1]}'
        adj=[c for c in ["Round","Sex","Breed","Member","weight","recording_duration"] if c in tmp and c not in [a,b] and tmp[c].nunique()>1]
        adj_terms=[f'C(Q("{c}"))' if tmp[c].dtype=="object" else f'Q("{c}")' for c in adj]
        formula=f'Q("{outcome}") ~ '+" + ".join(terms+[interaction]+adj_terms)
        try:
            fit=ols(formula,tmp).fit(cov_type="cluster",cov_kwds={"groups":tmp["AnimalID"]})
            interaction_rows.append({"phenotype":outcome,"interaction":f"{a} x {b}","n":len(tmp),
                "n_animals":tmp.AnimalID.nunique(),"model_p_value":fit.f_pvalue,
                "model_r_squared":fit.rsquared,"formula":formula})
        except Exception as e:
            interaction_rows.append({"phenotype":outcome,"interaction":f"{a} x {b}","error":str(e)[:200]})
interaction_df=pd.DataFrame(interaction_rows)
if len(interaction_df):
    valid=interaction_df["model_p_value"].notna()
    if valid.any():
        interaction_df.loc[valid,"model_p_value_FDR"]=multipletests(interaction_df.loc[valid,"model_p_value"],method="fdr_bh")[1]
display(interaction_df)
interaction_df.to_csv(TAB/"biological_interaction_models.csv",index=False)


# 28. Integrated methane phenotype evaluation scorecard

This section consolidates evidence from repeatability, duration robustness, body-size scaling, model predictability and biological interpretation. It deliberately reports component metrics separately. A weighted composite score should only be constructed after prespecifying weights and confirming that each component is measured on a comparable scale.


In [ ]:
scorecard_rows=[]
for outcome in [c for c in ["mean_methane","mean_resp","mean_eruct","max_resp","max_eruct",
                            "eruct_per_min","AUC_peak_per_sec","AUC_per_sec"] if c in df.columns]:
    row={"phenotype":outcome}
    # Existing ICC results, where available.
    if "icc_df" in globals() and isinstance(icc_df,pd.DataFrame) and len(icc_df):
        icc_col=next((c for c in ["phenotype","variable","outcome"] if c in icc_df.columns),None)
        if icc_col:
            hit=icc_df[icc_df[icc_col].astype(str)==outcome]
            if len(hit):
                for c in hit.columns:
                    if "icc" in c.lower(): row["repeatability_"+c]=hit.iloc[0][c]
    # Animal-level within-animal CV.
    if "AnimalID" in df:
        tmp=df[["AnimalID",outcome]].copy()
        tmp[outcome]=pd.to_numeric(tmp[outcome],errors="coerce")
        g=tmp.groupby("AnimalID")[outcome]
        means, sds=g.mean(),g.std()
        cv=(sds/means.abs().replace(0,np.nan))*100
        row["median_within_animal_CV_percent"]=cv.replace([np.inf,-np.inf],np.nan).median()
        row["animals_with_repeats"]=(g.size()>1).sum()
    # Duration association as a diagnostic, not a desirability score.
    if duration_col:
        z=df[[outcome,duration_col]].apply(pd.to_numeric,errors="coerce").dropna()
        if len(z)>10:
            rho,p=stats.spearmanr(z[outcome],z[duration_col])
            row["duration_spearman_rho"]=rho
            row["duration_spearman_p"]=p
    # Allometric slope.
    if len(allometry_df) and outcome in allometry_df.phenotype.values:
        ar=allometry_df[allometry_df.phenotype==outcome].iloc[0]
        row["allometric_weight_slope"]=ar.get("log_log_slope",np.nan)
        row["allometric_slope_p"]=ar.get("p_value",np.nan)
    scorecard_rows.append(row)
phenotype_scorecard=pd.DataFrame(scorecard_rows)
display(phenotype_scorecard)
phenotype_scorecard.to_csv(TAB/"integrated_methane_phenotype_scorecard.csv",index=False)

print("Scientific extension complete. Review all diagnostics, convergence messages, sample sizes and model assumptions before manuscript reporting.")


## 29. Additional reporting and interpretation safeguards

1. Distinguish sensor-derived methane indices from calibrated methane mass flux. Do not report g/day, methane yield or emission intensity unless supported by calibration and appropriate denominators.
2. Repeatability is not heritability. Genetic parameters require pedigree or genomic relationship information and an appropriate animal model.
3. Treat CBBP/non-CBBP results as adjusted observational associations unless a causal design and confounding strategy justify causal interpretation.
4. Do not interpret cluster labels as objectively validated biological classes without independent replication, stability and biological validation.
5. Correct families of hypothesis tests for multiplicity and report effect sizes with confidence intervals, not p-values alone.
6. Report missingness, exclusions, convergence failures, model specifications, software versions and validation split design.
7. Avoid using any animal-level information from validation animals during preprocessing, feature selection, tuning or threshold selection.
8. The phenotype scorecard is a decision aid, not an automatic ranking. Select the primary phenotype based on prespecified biological rationale and evidence across all dimensions.
9. If pedigree/genomic data become available, add animal-model heritability and genetic correlation analyses as a separate genetic evaluation module.
10. Preserve the distinction between inference (GEE/LMM/OLS), prediction (grouped CV), and unsupervised discovery (clustering/PCA); these answer different scientific questions.
